# PatchCore Reference — pose A training and efficiency

Standalone Kaggle run for **A_overhead only**, using the predefined **720 × 720 pose crop**,
WideResNet-50-2 and a **50,000-entry memory bank**. Training means fitting a clean-only
memory bank and calibration; there is no gradient-based optimization.

Attach the extracted synthetic dataset (`samples.csv`, `images/`, `masks/`), enable a GPU
(prefer the same T4 used for other model benchmarks), and enable Internet for ImageNet
weights. Defaults select only pose A in **train, validation and test**. The code below is an
explicit copy of the project modules, so the repository is not needed on Kaggle.

The Reference geometry is preserved **after** the pose crop: resize shorter side to 256,
then center-crop to 224 × 224 and normalize with ImageNet statistics. As in the existing
notebooks, the default pre-fit audit is visual and non-blocking. Optional numerical mask
diagnostics can report clipped or empty defects, but never stop the fit or remove samples.
The Reference center crop can exclude edge defects; the original labels are preserved.
Metrics are not used to select a new configuration.

**Fit cost:** the candidate reservoir can reach 1,000,000 × 1,024 float32 embeddings.
Greedy coreset selection of 50,000 entries is expensive and requires substantial host RAM.
Use a full GPU session, not a CPU session or a reduced smoke-test bank. The per-image
sampling cap is 512 rather than 128: with 2,800 pose-A training images and coreset ratio 0.1,
128 would yield only 35,840 entries. Both the requested and actual bank sizes are saved.

GPU energy is measured after the latency benchmark, using the same separate NVML protocol as EfficientAD. In an existing fitted session, the new energy cells can run without repeating the fit.


In [ ]:
from __future__ import annotations

# Kaggle already includes the local ML dependencies.
# Only if enabling DRIVE_UPLOAD_ENABLED below, uncomment and execute:
# %pip install -q google-api-python-client google-auth tqdm


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import threading
import time
from collections import Counter, deque
from collections.abc import Iterable, Mapping
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import pil_to_tensor

from torchvision.transforms.functional import gaussian_blur
from tqdm import tqdm


In [ ]:
from __future__ import annotations

# Configuration
KAGGLE_INPUT = Path("/kaggle/input")
BATCH_SIZE = 4
NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PatchCore presets: Amazon-compatible reference and project-efficient light.
MODEL_NAME = "patchcore"
PATCHCORE_PRESET = "reference"  # ["light", "reference"]
PATCHCORE_PRESETS = {
    "light": {
        "backbone": "resnet18", "input_size": (384, 512),
        "resize": (384, 512), "resize_shorter_side": None, "center_crop": None,
        "num_neighbors": 9, "pretrain_embed_dimension": 384,
        "target_embed_dimension": 384, "projection_dim": 64, "sampling_seed": SEED,
    },
    "reference": {
        "backbone": "wide_resnet50_2", "input_size": (224, 224),
        "resize": None, "resize_shorter_side": 256, "center_crop": (224, 224),
        "num_neighbors": 1, "pretrain_embed_dimension": 1024,
        "target_embed_dimension": 1024, "projection_dim": 128, "sampling_seed": 0,
    },
}
if PATCHCORE_PRESET not in PATCHCORE_PRESETS:
    raise ValueError("PATCHCORE_PRESET must be one of: light, reference")
PATCHCORE_CONFIG = PATCHCORE_PRESETS[PATCHCORE_PRESET]
PATCHCORE_BACKBONE = PATCHCORE_CONFIG["backbone"]
PATCHCORE_PRETRAINED = True
PATCHCORE_LAYERS = ("layer2", "layer3")
PATCHCORE_CORESET_SAMPLING_RATIO = 0.1
PATCHCORE_NUM_NEIGHBORS = PATCHCORE_CONFIG["num_neighbors"]
PATCHCORE_PATCH_SIZE = 3
PATCHCORE_PATCH_STRIDE = 1
PATCHCORE_PRETRAIN_EMBED_DIMENSION = PATCHCORE_CONFIG["pretrain_embed_dimension"]
PATCHCORE_TARGET_EMBED_DIMENSION = PATCHCORE_CONFIG["target_embed_dimension"]
PATCHCORE_MAX_PATCHES_PER_IMAGE = 512
PATCHCORE_MAX_TRAINING_EMBEDDINGS = 1_000_000
PATCHCORE_MAX_MEMORY_BANK_SIZE = 50_000
PATCHCORE_PROJECTION_DIM = PATCHCORE_CONFIG["projection_dim"]
PATCHCORE_SAMPLING_SEED = PATCHCORE_CONFIG["sampling_seed"]
PATCHCORE_CALIBRATION_QUANTILE = 0.99
PATCHCORE_CALIBRATION_BATCHES = 20
PATCHCORE_DISTANCE_QUERY_CHUNK_SIZE = 512
PATCHCORE_DISTANCE_BANK_CHUNK_SIZE = 2048
PATCHCORE_GAUSSIAN_SIGMA = 4.0
IMAGE_SIZE = PATCHCORE_CONFIG["input_size"]
PREPROCESS_RESIZE = PATCHCORE_CONFIG["resize"]
PREPROCESS_RESIZE_SHORTER_SIDE = PATCHCORE_CONFIG["resize_shorter_side"]
PREPROCESS_CENTER_CROP = PATCHCORE_CONFIG["center_crop"]
MODEL_RUN_NAME = (
    f"patchcore_{PATCHCORE_PRESET}_{PATCHCORE_BACKBONE}_"
    f"{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_A_crop720_bank50k"
)

# PatchCore fits a memory bank without gradient-based optimization.
OPTIMIZER_NAME = "none"
SCHEDULER_NAME = "none"

# Fixed pose crop, followed by the Reference resize/center-crop geometry.
CAMERA_POSES = ("A_overhead",)
POSE_CROP_ENABLED = True
POSE_CROPS = {"A_overhead": (90, 240, 720, 720)}
EVALUATION_BATCH_SIZE = 1
DATASET_ROOT_OVERRIDE = None  # Optional explicit Path to the extracted dataset.
REQUIRE_CUDA = True  # Avoid accidentally starting the full 50k fit on CPU.
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)
TRAIN_AUGMENTATIONS_ENABLED = False
TRAIN_BRIGHTNESS = 0.08
TRAIN_CONTRAST = 0.08
TRAIN_GAMMA = 0.08
TRAIN_SATURATION = 0.06
TRAIN_SENSOR_NOISE = 0.004
TRAIN_GAUSSIAN_NOISE = 0.004
TRAIN_GAUSSIAN_BLUR = (0.1, 0.5)

# Threshold-free evaluation metrics.
METRIC_NAMES = (
    "image_auroc",
    "image_average_precision",
    "pixel_auroc",
    "pixel_average_precision",
)
METRIC_HISTOGRAM_BINS = 2048
RESTRICT_PIXELS_TO_TARGET_MASK = False

# Extended diagnostics and failure analysis.
DIAGNOSTICS_ENABLED = True
DIAGNOSTICS_GROUP_FIELDS = ("severity", "camera_pose", "lighting", "wear")
DIAGNOSTICS_NUM_EXTREME_EXAMPLES = 6
DIAGNOSTICS_PRO_BINS = 256
DIAGNOSTICS_PRO_MAX_FPR = 0.30

# Qualitative prediction visualization.
VISUALIZATION_ENABLED = True
VISUALIZATION_NUM_CLEAN = 3
VISUALIZATION_NUM_ANOMALOUS = 3
VISUALIZATION_COLORMAP = "magma"
VISUALIZATION_OVERLAY_ALPHA = 0.55
VISUALIZATION_DPI = 150

# Optional Google Drive persistence through Kaggle Secrets.
DRIVE_UPLOAD_ENABLED = False
DRIVE_PARENT_FOLDER_ID = None
DRIVE_UPLOAD_CHECKPOINTS = True
RUN_ID = (
    f"{MODEL_RUN_NAME}__seed{SEED}__"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
)
OUTPUT_DIR = Path("/kaggle/working") / RUN_ID

# Same non-blocking visual preprocessing audit as the existing notebooks.
# Optional numerical diagnostics: never filter samples or stop the fit.
MASK_VISIBILITY_AUDIT_ENABLED = False
AUDIT_ENABLED = True
AUDIT_SPLIT = "validation"
AUDIT_INDICES = None  # Automatically select one clean and one hole example.
AUDIT_VARIANTS = 1

# Expected pose-A subset of the full v1_10000 manifest.
EXPECTED_SPLIT_CONDITION_COUNTS = Counter({
    ("train", "clean"): 2800,
    ("validation", "clean"): 296,
    ("validation", "hole"): 96,
    ("test", "clean"): 404,
    ("test", "hole"): 404,
})
# Additional audit outputs, separate from the four primary metrics.
AUDIT_METRIC_NAMES = (
    "target_wheel_pixel_auroc",
    "target_wheel_pixel_average_precision",
    "normalized_image_scores_at_zero",
    "normalized_image_scores_at_one",
    "normalized_image_score_saturation_fraction",
)

# Batch-one efficiency. No model changes, gradients, or metric-based selection.
BENCHMARK_ENABLED = True
BENCHMARK_WARMUP_RUNS = 10
BENCHMARK_TIMED_RUNS = 50
BENCHMARK_SPLIT = "validation"

# Separate sustained GPU energy measurement; same settings as EfficientAD.
ENERGY_ENABLED = True
ENERGY_METHOD = "auto"  # NVML total-energy counter, or measured-power integration
ENERGY_REPEATS = 3
ENERGY_DURATION_SECONDS = 30.0
ENERGY_WARMUP_SECONDS = 5.0
ENERGY_POWER_SAMPLE_INTERVAL_SECONDS = 0.1


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

# Optional sensor dependency. Works in an older live session too.
# Installation/sensor failure never fabricates an energy estimate.
if globals().get("ENERGY_ENABLED", True) and importlib.util.find_spec("pynvml") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py"])
    except Exception as error:
        print(f"WARNING: NVML bindings could not be installed; energy may be unavailable: {error}")


## Definitions

The following cells only define reusable classes and functions; they do not start the experiment.

In [ ]:
from __future__ import annotations

RGBTriplet = tuple[float, float, float]
ImageSize = tuple[int, int]
SigmaRange = tuple[float, float]
CropBox = tuple[int, int, int, int]
IMAGENET_MEAN: RGBTriplet = (0.485, 0.456, 0.406)
IMAGENET_STD: RGBTriplet = (0.229, 0.224, 0.225)


@dataclass(frozen=True)
class PreprocessingConfig:
    """Optional preprocessing and train-time image augmentation settings.

    ``resize`` follows the torchvision convention ``(height, width)``. Every
    scalar augmentation value is disabled when ``None`` and otherwise denotes
    its maximum intensity. Photometric transforms affect RGB only.
    """

    resize: ImageSize | None = None
    resize_shorter_side: int | None = None
    center_crop: ImageSize | None = None
    pose_crop_enabled: bool = False
    pose_crops: Mapping[str, CropBox | None] = field(default_factory=dict)
    normalize_mean: RGBTriplet | None = None
    normalize_std: RGBTriplet | None = None
    augmentations_enabled: bool = True
    brightness: float | None = None
    contrast: float | None = None
    gamma: float | None = None
    saturation: float | None = None
    sensor_noise: float | None = None
    gaussian_noise: float | None = None
    gaussian_blur: SigmaRange | None = None

    def __post_init__(self) -> None:
        if self.resize is not None and self.resize_shorter_side is not None:
            raise ValueError("resize and resize_shorter_side are mutually exclusive")
        if self.resize is not None:
            if len(self.resize) != 2 or any(value < 1 for value in self.resize):
                raise ValueError("resize must be a positive (height, width) pair")
        if self.resize_shorter_side is not None and self.resize_shorter_side < 1:
            raise ValueError("resize_shorter_side must be positive")
        if self.center_crop is not None:
            if len(self.center_crop) != 2 or any(value < 1 for value in self.center_crop):
                raise ValueError("center_crop must be a positive (height, width) pair")
        for pose, crop in self.pose_crops.items():
            if not pose:
                raise ValueError("pose crop names cannot be empty")
            if crop is None:
                continue
            if len(crop) != 4:
                raise ValueError("pose crops must be (top, left, height, width)")
            if crop[2] < 1 or crop[3] < 1:
                raise ValueError("pose crop height and width must be positive")
            if crop[2] != crop[3]:
                raise ValueError("pose crops must be square")

        if (self.normalize_mean is None) != (self.normalize_std is None):
            raise ValueError("normalize_mean and normalize_std must be set together")
        if self.normalize_mean is not None:
            if len(self.normalize_mean) != 3 or len(self.normalize_std or ()) != 3:
                raise ValueError("normalization mean and std must contain three RGB values")
            if any(value <= 0 for value in self.normalize_std or ()):
                raise ValueError("normalization std values must be positive")

        for name in (
            "brightness",
            "contrast",
            "gamma",
            "saturation",
            "sensor_noise",
            "gaussian_noise",
        ):
            value = getattr(self, name)
            if value is not None and value < 0:
                raise ValueError(f"{name} must be non-negative or None")

        for name in ("brightness", "contrast", "gamma", "saturation"):
            value = getattr(self, name)
            if value is not None and value >= 1:
                raise ValueError(f"{name} must be smaller than 1")

        if self.gaussian_blur is not None:
            if len(self.gaussian_blur) != 2:
                raise ValueError("gaussian_blur must be a (min_sigma, max_sigma) pair")
            minimum, maximum = self.gaussian_blur
            if minimum <= 0 or maximum < minimum:
                raise ValueError("gaussian_blur requires 0 < min_sigma <= max_sigma")


class WheelPreprocessor:
    """Apply aligned geometric transforms and RGB-only photometric transforms."""

    def __init__(self, config: PreprocessingConfig | None = None) -> None:
        self.config = config or PreprocessingConfig()

    @staticmethod
    def _symmetric_factor(maximum_delta: float) -> float:
        return 1.0 + (2.0 * torch.rand(1).item() - 1.0) * maximum_delta

    @staticmethod
    def _odd_kernel_size(maximum_sigma: float) -> int:
        radius = max(1, round(3.0 * maximum_sigma))
        return 2 * radius + 1

    def __call__(
        self,
        image: torch.Tensor,
        target_mask: torch.Tensor,
        anomaly_mask: torch.Tensor,
        *,
        camera_pose: str | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        config = self.config

        if config.pose_crop_enabled:
            if camera_pose is None:
                raise ValueError(
                    "camera_pose is required when pose cropping is enabled"
                )
            if camera_pose not in config.pose_crops:
                raise ValueError(f"No pose crop configured for {camera_pose!r}")
            crop = config.pose_crops[camera_pose]
            if crop is not None:
                top, left, height, width = crop
                image_height, image_width = image.shape[-2:]
                if height > image_height or width > image_width:
                    raise ValueError(
                        f"Pose crop {crop!r} for {camera_pose!r} cannot fit "
                        f"inside the input image {(image_height, image_width)!r} "
                        "without padding"
                    )
                top = min(max(top, 0), image_height - height)
                left = min(max(left, 0), image_width - width)
                image = TF.crop(image, top, left, height, width)
                target_mask = TF.crop(target_mask, top, left, height, width)
                anomaly_mask = TF.crop(anomaly_mask, top, left, height, width)

        if config.resize is not None:
            image = TF.resize(
                image,
                config.resize,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
        elif config.resize_shorter_side is not None:
            image = TF.resize(
                image,
                config.resize_shorter_side,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )

        if config.center_crop is not None:
            image = TF.center_crop(image, config.center_crop)
            target_mask = TF.center_crop(target_mask, config.center_crop)
            anomaly_mask = TF.center_crop(anomaly_mask, config.center_crop)

        # Every model-facing split uses one stable numerical contract.  Keeping
        # uint8 for evaluation while train-time augmentation returned float32
        # would either fail in the backbone or change the input scale by 255x.
        image = TF.convert_image_dtype(image, torch.float32)

        if config.augmentations_enabled:
            if config.brightness is not None:
                image = TF.adjust_brightness(image, self._symmetric_factor(config.brightness))
            if config.contrast is not None:
                image = TF.adjust_contrast(image, self._symmetric_factor(config.contrast))
            if config.gamma is not None:
                image = TF.adjust_gamma(image, self._symmetric_factor(config.gamma))
            if config.saturation is not None:
                image = TF.adjust_saturation(image, self._symmetric_factor(config.saturation))

            if config.gaussian_blur is not None:
                minimum, maximum = config.gaussian_blur
                sigma = minimum + torch.rand(1).item() * (maximum - minimum)
                kernel_size = self._odd_kernel_size(maximum)
                image = TF.gaussian_blur(image, kernel_size, sigma)

            if config.sensor_noise is not None and config.sensor_noise > 0:
                signal_scale = image.clamp(0.0, 1.0).sqrt()
                image = image + torch.randn_like(image) * signal_scale * config.sensor_noise
            if config.gaussian_noise is not None and config.gaussian_noise > 0:
                image = image + torch.randn_like(image) * config.gaussian_noise

        image = image.clamp(0.0, 1.0)
        if config.normalize_mean is not None:
            image = TF.normalize(
                image,
                mean=config.normalize_mean,
                std=config.normalize_std,
            )

        return image, target_mask, anomaly_mask

    def image_for_display(self, image: torch.Tensor) -> torch.Tensor:
        """Return a float RGB image in [0, 1], undoing normalization if needed."""
        if image.dtype == torch.uint8:
            return TF.convert_image_dtype(image, torch.float32)

        result = image.detach().clone()
        if self.config.normalize_mean is not None:
            mean = result.new_tensor(self.config.normalize_mean).view(3, 1, 1)
            std = result.new_tensor(self.config.normalize_std).view(3, 1, 1)
            result = result * std + mean
        return result.clamp(0.0, 1.0)


In [ ]:
from __future__ import annotations

# Kaggle exposes attached datasets as extracted directories.
def find_dataset_root(input_root: Path = KAGGLE_INPUT) -> Path:
    candidates = [
        manifest.parent
        for manifest in input_root.rglob("samples.csv")
        if (manifest.parent / "images").is_dir()
        and (manifest.parent / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No valid dataset found under {input_root}: "
            "samples.csv, images/, and masks/ are required."
        )
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple dataset candidates found: {candidates}")
    return candidates[0]

In [ ]:
from __future__ import annotations

REQUIRED_COLUMNS = {
    "image_id",
    "split",
    "condition",
    "image_path",
    "target_mask_path",
    "anomaly_mask_path",
    "pair_id",
    "camera_pose",
}
VALID_SPLITS = ("train", "validation", "test")
VALID_CONDITIONS = {"clean", "hole"}
ARTIFACT_PATH_FIELDS = ("image_path", "target_mask_path", "anomaly_mask_path")


class CuriosityWheelDataset(Dataset[dict[str, Any]]):
    """Load one split of the extracted Curiosity wheel Kaggle dataset."""

    def __init__(
        self,
        root: str | Path,
        split: str,
        preprocessing: WheelPreprocessor | None = None,
        camera_poses: Sequence[str] | None = None,
    ) -> None:
        self.root = Path(root).expanduser().resolve()
        self.split = split
        self.preprocessing = preprocessing or WheelPreprocessor()

        if split not in VALID_SPLITS:
            raise ValueError(f"Unsupported split {split!r}; expected one of {VALID_SPLITS}")
        if not self.root.is_dir():
            raise NotADirectoryError(f"Dataset root is not a directory: {self.root}")

        manifest_path = self.root / "samples.csv"
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Dataset manifest is missing: {manifest_path}")

        with manifest_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            columns = set(reader.fieldnames or [])
            missing = REQUIRED_COLUMNS - columns
            if missing:
                raise ValueError(f"samples.csv is missing required columns: {sorted(missing)}")
            rows = list(reader)

        self._validate_manifest_rows(rows)
        if isinstance(camera_poses, str):
            raise TypeError("camera_poses must be a sequence of pose names, not a string")
        if camera_poses is None:
            self.camera_poses = None
        else:
            self.camera_poses = tuple(dict.fromkeys(str(pose) for pose in camera_poses))
            if not self.camera_poses or any(not pose for pose in self.camera_poses):
                raise ValueError("camera_poses must contain at least one non-empty pose")
            available_poses = {row.get("camera_pose", "") for row in rows}
            unknown_poses = set(self.camera_poses) - available_poses
            if unknown_poses:
                raise ValueError(
                    f"Unknown camera poses: {sorted(unknown_poses)}; "
                    f"available poses: {sorted(available_poses)}"
                )

        self.rows = [
            row
            for row in rows
            if row["split"] == split
            and (
                self.camera_poses is None
                or row.get("camera_pose") in self.camera_poses
            )
        ]

        if not self.rows:
            raise ValueError(
                f"No samples found for split {split!r} and "
                f"camera_poses={self.camera_poses!r}"
            )

        for row in self.rows:
            for field in ARTIFACT_PATH_FIELDS:
                relative_path = row[field]
                if relative_path and not (self.root / relative_path).is_file():
                    raise FileNotFoundError(
                        f"Dataset artifact is missing: {self.root / relative_path}"
                    )

    @classmethod
    def _validate_manifest_rows(cls, rows: list[dict[str, str]]) -> None:
        """Validate the complete manifest before selecting one split."""
        if not rows:
            raise ValueError("samples.csv contains no samples")

        image_ids: set[str] = set()
        pairs: dict[str, list[dict[str, str]]] = {}
        for row in rows:
            image_id = row["image_id"]
            if not image_id:
                raise ValueError("samples.csv contains an empty image_id")
            if image_id in image_ids:
                raise ValueError(f"Duplicate image_id in samples.csv: {image_id}")
            image_ids.add(image_id)

            row_split = row["split"]
            if row_split not in VALID_SPLITS:
                raise ValueError(
                    f"Unsupported split {row_split!r} in sample {image_id}; "
                    f"expected one of {VALID_SPLITS}"
                )

            condition = row["condition"]
            if condition not in VALID_CONDITIONS:
                raise ValueError(f"Unsupported condition {condition!r} in sample {image_id}")
            if not row["image_path"] or not row["target_mask_path"]:
                raise ValueError(f"Sample has incomplete artifact paths: {image_id}")
            if condition == "hole" and not row["anomaly_mask_path"]:
                raise ValueError(f"Hole sample has no anomaly mask: {image_id}")
            if condition == "clean" and row["anomaly_mask_path"]:
                raise ValueError(f"Clean sample unexpectedly has an anomaly mask: {image_id}")
            for field in ARTIFACT_PATH_FIELDS:
                if row[field]:
                    cls._validate_relative_path(row[field])

            pair_id = row["pair_id"]
            if condition == "hole" and not pair_id:
                raise ValueError(f"Hole sample has no pair_id: {image_id}")
            if pair_id:
                pairs.setdefault(pair_id, []).append(row)

        for pair_id, pair_rows in pairs.items():
            conditions = {row["condition"] for row in pair_rows}
            splits = {row["split"] for row in pair_rows}
            if len(pair_rows) != 2 or conditions != VALID_CONDITIONS:
                raise ValueError(
                    f"Pair {pair_id!r} must contain exactly one clean and one hole sample"
                )
            if len(splits) != 1:
                raise ValueError(f"Pair {pair_id!r} crosses dataset splits: {sorted(splits)}")

    @staticmethod
    def _validate_relative_path(value: str) -> None:
        path = Path(value)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Invalid dataset-relative path: {value!r}")

    def _load_image(self, relative_path: str, mode: str) -> Image.Image:
        path = self.root / relative_path
        if not path.is_file():
            raise FileNotFoundError(f"Dataset artifact is missing: {path}")
        with Image.open(path) as image:
            return image.convert(mode)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        image = pil_to_tensor(self._load_image(row["image_path"], "RGB"))
        target_mask = pil_to_tensor(self._load_image(row["target_mask_path"], "L"))

        if image.shape[1:] != target_mask.shape[1:]:
            raise ValueError(f"Image/target mask size mismatch for {row['image_id']}")

        anomaly_path = row["anomaly_mask_path"]
        if anomaly_path:
            anomaly_mask = pil_to_tensor(self._load_image(anomaly_path, "L"))
            if anomaly_mask.shape != target_mask.shape:
                raise ValueError(f"Target/anomaly mask size mismatch for {row['image_id']}")
        else:
            # Clean samples use an empty mask so every batch has one stable schema.
            anomaly_mask = torch.zeros_like(target_mask)

        image, target_mask, anomaly_mask = self.preprocessing(
            image,
            target_mask,
            anomaly_mask,
            camera_pose=row.get("camera_pose"),
        )

        return {
            "image": image,
            "target_mask": target_mask,
            "anomaly_mask": anomaly_mask,
            "label": torch.tensor(row["condition"] == "hole", dtype=torch.long),
            "has_anomaly_mask": bool(anomaly_path),
            "metadata": {
                key: value
                for key, value in row.items()
                if key not in {"image_path", "target_mask_path", "anomaly_mask_path"}
            },
        }


In [ ]:
from __future__ import annotations

def build_dataloader(
    root: str | Path,
    split: str,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    preprocessing: WheelPreprocessor | None = None,
    camera_poses: tuple[str, ...] | None = None,
) -> DataLoader:
    """Build one loader; only the training split is shuffled."""
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    if num_workers < 0:
        raise ValueError("num_workers cannot be negative")

    dataset = CuriosityWheelDataset(
        root,
        split,
        preprocessing=preprocessing,
        camera_poses=camera_poses,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=split == "train",
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        generator=torch.Generator().manual_seed(seed),
    )


In [ ]:
from __future__ import annotations

CHECKPOINT_SCHEMA_VERSION = 2


@dataclass(frozen=True)
class AnomalyPrediction:
    """Model-independent anomaly output with normalized scores."""

    anomaly_score: torch.Tensor
    anomaly_map: torch.Tensor

    def __post_init__(self) -> None:
        if self.anomaly_score.ndim != 1:
            raise ValueError("anomaly_score must have shape [B]")
        if self.anomaly_map.ndim != 4 or self.anomaly_map.shape[1] != 1:
            raise ValueError("anomaly_map must have shape [B, 1, H, W]")
        if self.anomaly_score.shape[0] != self.anomaly_map.shape[0]:
            raise ValueError("anomaly_score and anomaly_map batch sizes must match")
        for name, value in (
            ("anomaly_score", self.anomaly_score),
            ("anomaly_map", self.anomaly_map),
        ):
            if not value.is_floating_point() or not torch.isfinite(value).all():
                raise ValueError(f"{name} must contain finite floating-point values")
            if value.numel() and (value.min() < 0 or value.max() > 1):
                raise ValueError(f"{name} must be normalized to [0, 1]")


class AnomalyDetector(nn.Module):
    """Small common interface shared by all anomaly detectors."""

    @property
    def is_fitted(self) -> bool:
        raise NotImplementedError

    def fit(
        self,
        train_loader: Iterable[Mapping[str, Any]],
        *,
        device: torch.device | str,
        validation_loader: Iterable[Mapping[str, Any]] | None = None,
        work_dir: str | Path | None = None,
        resume: bool = False,
    ) -> AnomalyDetector:
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        raise NotImplementedError

    def checkpoint_config(self) -> dict[str, Any]:
        """Return behavior-affecting constructor settings for strict reloads."""
        return {}

    def save(self, path: str | Path, *, metadata: Mapping[str, Any] | None = None) -> Path:
        """Save model state and small run metadata in one checkpoint."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "schema_version": CHECKPOINT_SCHEMA_VERSION,
                "model_class": f"{type(self).__module__}.{type(self).__qualname__}",
                "model_config": self.checkpoint_config(),
                "model_state_dict": self.state_dict(),
                "metadata": dict(metadata or {}),
            },
            path,
        )
        return path

    def _prepare_state_dict_for_load(self, state_dict: Mapping[str, torch.Tensor]) -> None:
        """Allow models with dynamically sized buffers to prepare before load."""

    def load(
        self,
        path: str | Path,
        *,
        map_location: torch.device | str | None = None,
    ) -> dict[str, Any]:
        """Load a trusted project checkpoint and return its metadata."""
        payload = torch.load(Path(path), map_location=map_location)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            raise ValueError("Checkpoint does not contain model_state_dict")
        if payload.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
            raise ValueError(
                f"Unsupported checkpoint schema {payload.get('schema_version')!r}; "
                f"expected {CHECKPOINT_SCHEMA_VERSION}"
            )
        expected_class = f"{type(self).__module__}.{type(self).__qualname__}"
        if payload.get("model_class") != expected_class:
            raise ValueError(
                f"Checkpoint model class {payload.get('model_class')!r} does not "
                f"match {expected_class!r}"
            )
        saved_config = payload.get("model_config")
        expected_config = self.checkpoint_config()
        if saved_config != expected_config:
            raise ValueError(
                "Checkpoint model configuration does not match the current model: "
                f"saved={saved_config!r}, current={expected_config!r}"
            )
        state_dict = payload["model_state_dict"]
        self._prepare_state_dict_for_load(state_dict)
        self.load_state_dict(state_dict)
        return dict(payload.get("metadata", {}))


In [ ]:
from __future__ import annotations

class PatchCore(AnomalyDetector):
    """Memory-bounded PatchCore anomaly detector with a frozen backbone."""

    def __init__(
        self,
        *,
        backbone: str = "resnet18",
        pretrained: bool = True,
        layers: Sequence[str] = ("layer2", "layer3"),
        coreset_sampling_ratio: float = 0.1,
        num_neighbors: int = 9,
        patch_size: int = 3,
        patch_stride: int = 1,
        pretrain_embed_dimension: int = 384,
        target_embed_dimension: int = 384,
        max_patches_per_image: int = 128,
        max_training_embeddings: int = 50_000,
        max_memory_bank_size: int = 2_048,
        projection_dim: int = 64,
        sampling_seed: int = 42,
        calibration_quantile: float = 0.99,
        calibration_batches: int = 20,
        distance_query_chunk_size: int = 512,
        distance_bank_chunk_size: int = 2_048,
        gaussian_sigma: float = 4.0,
    ) -> None:
        super().__init__()
        if not layers:
            raise ValueError("layers must contain at least one feature node")
        if not 0 < coreset_sampling_ratio <= 1:
            raise ValueError("coreset_sampling_ratio must be in (0, 1]")
        if num_neighbors < 1:
            raise ValueError("num_neighbors must be at least 1")
        if patch_size < 1 or patch_size % 2 == 0:
            raise ValueError("patch_size must be a positive odd integer")
        for name, value in (
            ("max_patches_per_image", max_patches_per_image),
            ("max_training_embeddings", max_training_embeddings),
            ("max_memory_bank_size", max_memory_bank_size),
            ("projection_dim", projection_dim),
            ("patch_stride", patch_stride),
            ("pretrain_embed_dimension", pretrain_embed_dimension),
            ("target_embed_dimension", target_embed_dimension),
            ("calibration_batches", calibration_batches),
            ("distance_query_chunk_size", distance_query_chunk_size),
            ("distance_bank_chunk_size", distance_bank_chunk_size),
        ):
            if value < 1:
                raise ValueError(f"{name} must be at least 1")
        if not 0 < calibration_quantile <= 1:
            raise ValueError("calibration_quantile must be in (0, 1]")
        if gaussian_sigma < 0:
            raise ValueError("gaussian_sigma cannot be negative")

        weights = get_model_weights(backbone).DEFAULT if pretrained else None
        backbone_model = get_model(backbone, weights=weights)
        self.feature_extractor = create_feature_extractor(
            backbone_model,
            return_nodes={layer: layer for layer in layers},
        )
        self.feature_extractor.requires_grad_(False)

        self.backbone = backbone
        self.pretrained = bool(pretrained)
        self.layers = tuple(layers)
        self.coreset_sampling_ratio = float(coreset_sampling_ratio)
        self.num_neighbors = int(num_neighbors)
        self.patch_size = int(patch_size)
        self.patch_stride = int(patch_stride)
        self.pretrain_embed_dimension = int(pretrain_embed_dimension)
        self.target_embed_dimension = int(target_embed_dimension)
        self.max_patches_per_image = int(max_patches_per_image)
        self.max_training_embeddings = int(max_training_embeddings)
        self.max_memory_bank_size = int(max_memory_bank_size)
        self.projection_dim = int(projection_dim)
        self.sampling_seed = int(sampling_seed)
        self.calibration_quantile = float(calibration_quantile)
        self.calibration_batches = int(calibration_batches)
        self.distance_query_chunk_size = int(distance_query_chunk_size)
        self.distance_bank_chunk_size = int(distance_bank_chunk_size)
        self.gaussian_sigma = float(gaussian_sigma)

        self.register_buffer("memory_bank", torch.empty(0, 0), persistent=True)
        self.register_buffer("image_score_scale", torch.tensor(0.0), persistent=True)
        self.register_buffer("anomaly_map_scale", torch.tensor(0.0), persistent=True)
        self.fit_summary: dict[str, int | float | list[int]] = {}
        self.train(False)

    @property
    def is_fitted(self) -> bool:
        return (
            self.memory_bank.ndim == 2
            and self.memory_bank.shape[0] > 0
            and self.image_score_scale.item() > 0
            and self.anomaly_map_scale.item() > 0
        )

    def checkpoint_config(self) -> dict[str, object]:
        return {
            "backbone": self.backbone,
            "layers": list(self.layers),
            "coreset_sampling_ratio": self.coreset_sampling_ratio,
            "num_neighbors": self.num_neighbors,
            "patch_size": self.patch_size,
            "patch_stride": self.patch_stride,
            "pretrain_embed_dimension": self.pretrain_embed_dimension,
            "target_embed_dimension": self.target_embed_dimension,
            "max_patches_per_image": self.max_patches_per_image,
            "max_training_embeddings": self.max_training_embeddings,
            "max_memory_bank_size": self.max_memory_bank_size,
            "projection_dim": self.projection_dim,
            "sampling_seed": self.sampling_seed,
            "calibration_quantile": self.calibration_quantile,
            "calibration_batches": self.calibration_batches,
            "distance_query_chunk_size": self.distance_query_chunk_size,
            "distance_bank_chunk_size": self.distance_bank_chunk_size,
            "gaussian_sigma": self.gaussian_sigma,
        }

    def train(self, mode: bool = True) -> PatchCore:
        """Keep the frozen backbone and its BatchNorm layers in evaluation mode."""
        super().train(False)
        return self

    @staticmethod
    def _validate_images(images: torch.Tensor) -> None:
        if images.ndim != 4 or images.shape[1] != 3:
            raise ValueError(f"Expected RGB images [B, 3, H, W], got {tuple(images.shape)}")
        if not images.is_floating_point() or not torch.isfinite(images).all():
            raise ValueError("images must contain finite floating-point values")

    def _patchify(
        self,
        feature: torch.Tensor,
    ) -> tuple[torch.Tensor, tuple[int, int]]:
        """Extract overlapping local patches without reducing their channels."""
        padding = self.patch_size // 2
        patches = F.unfold(
            feature,
            kernel_size=self.patch_size,
            stride=self.patch_stride,
            padding=padding,
        ).transpose(1, 2)
        height = (feature.shape[-2] + 2 * padding - self.patch_size) // self.patch_stride + 1
        width = (feature.shape[-1] + 2 * padding - self.patch_size) // self.patch_stride + 1
        patches = patches.reshape(
            feature.shape[0],
            height * width,
            feature.shape[1],
            self.patch_size,
            self.patch_size,
        )
        return patches, (height, width)

    @staticmethod
    def _align_patch_grid(
        patches: torch.Tensor,
        source_size: tuple[int, int],
        target_size: tuple[int, int],
    ) -> torch.Tensor:
        """Interpolate patch grids while preserving channel and patch axes."""
        if source_size == target_size:
            return patches
        batch_size, _, channels, patch_height, patch_width = patches.shape
        patches = patches.reshape(
            batch_size,
            *source_size,
            channels,
            patch_height,
            patch_width,
        ).permute(0, 3, 4, 5, 1, 2)
        patches = patches.reshape(-1, 1, *source_size)
        patches = F.interpolate(
            patches,
            size=target_size,
            mode="bilinear",
            align_corners=False,
        )
        return patches.reshape(
            batch_size,
            channels,
            patch_height,
            patch_width,
            *target_size,
        ).permute(0, 4, 5, 1, 2, 3).reshape(
            batch_size,
            target_size[0] * target_size[1],
            channels,
            patch_height,
            patch_width,
        )

    @staticmethod
    def _map_embedding_dimension(
        patches: torch.Tensor,
        output_dimension: int,
    ) -> torch.Tensor:
        """Apply the reference MeanMapper operation to every local patch."""
        batch_size, num_patches = patches.shape[:2]
        flattened = patches.reshape(batch_size * num_patches, 1, -1)
        mapped = F.adaptive_avg_pool1d(flattened, output_dimension)
        return mapped.reshape(batch_size, num_patches, output_dimension)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Return reference-style locally aggregated embeddings [B, D, h, w]."""
        self._validate_images(images)
        features = self.feature_extractor(images)
        patch_layers = [self._patchify(features[layer]) for layer in self.layers]
        target_size = patch_layers[0][1]
        mapped_layers = []
        for patches, patch_grid in patch_layers:
            patches = self._align_patch_grid(patches, patch_grid, target_size)
            mapped_layers.append(
                self._map_embedding_dimension(
                    patches,
                    self.pretrain_embed_dimension,
                )
            )
        embeddings = torch.cat(mapped_layers, dim=-1)
        embeddings = self._map_embedding_dimension(
            embeddings,
            self.target_embed_dimension,
        )
        return embeddings.transpose(1, 2).reshape(
            images.shape[0],
            self.target_embed_dimension,
            *target_size,
        )

    def _patch_embeddings(
        self,
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, tuple[int, int]]:
        embedding_map = self(images)
        height, width = embedding_map.shape[-2:]
        embeddings = embedding_map.permute(0, 2, 3, 1).flatten(1, 2).contiguous()
        return embeddings, (height, width)

    def _sample_image_patches(
        self,
        embeddings: torch.Tensor,
        generator: torch.Generator,
    ) -> torch.Tensor:
        batch_size, num_patches, embedding_dim = embeddings.shape
        selected_count = min(self.max_patches_per_image, num_patches)
        if selected_count == num_patches:
            return embeddings.reshape(-1, embedding_dim)
        random_keys = torch.rand((batch_size, num_patches), generator=generator)
        indices = random_keys.topk(selected_count, dim=1, largest=False).indices
        indices = indices.to(embeddings.device)
        return embeddings.gather(
            1,
            indices.unsqueeze(-1).expand(-1, -1, embedding_dim),
        ).reshape(-1, embedding_dim)

    @staticmethod
    def _reduce_priority_reservoir(
        embedding_chunks: list[torch.Tensor],
        priority_chunks: list[torch.Tensor],
        maximum_size: int,
    ) -> tuple[list[torch.Tensor], list[torch.Tensor], int]:
        embeddings = torch.cat(embedding_chunks)
        priorities = torch.cat(priority_chunks)
        if embeddings.shape[0] > maximum_size:
            indices = priorities.topk(maximum_size, largest=True, sorted=False).indices
            embeddings = embeddings[indices]
            priorities = priorities[indices]
        return [embeddings], [priorities], embeddings.shape[0]

    def _build_coreset(
        self,
        candidates: torch.Tensor,
        *,
        device: torch.device,
        generator: torch.Generator,
    ) -> torch.Tensor:
        target_size = max(1, math.ceil(candidates.shape[0] * self.coreset_sampling_ratio))
        target_size = min(target_size, self.max_memory_bank_size, candidates.shape[0])
        if target_size == candidates.shape[0]:
            return candidates.contiguous()

        projection_dim = min(self.projection_dim, candidates.shape[1])
        projection = torch.randn(
            candidates.shape[1],
            projection_dim,
            generator=generator,
        ) / math.sqrt(projection_dim)
        projected = candidates.to(device) @ projection.to(device)

        first_index = int(torch.randint(candidates.shape[0], (1,), generator=generator))
        selected = torch.empty(target_size, dtype=torch.long, device=device)
        selected_mask = torch.zeros(candidates.shape[0], dtype=torch.bool, device=device)
        minimum_distances = torch.full(
            (candidates.shape[0],),
            torch.inf,
            dtype=projected.dtype,
            device=device,
        )
        current_index = first_index
        for position in tqdm(range(target_size), desc="PatchCore coreset", leave=False):
            selected[position] = current_index
            selected_mask[current_index] = True
            if position + 1 == target_size:
                break
            center = projected[current_index]
            distances = (projected - center).square().sum(dim=1)
            minimum_distances = torch.minimum(minimum_distances, distances)
            minimum_distances[selected_mask] = -1
            current_index = int(minimum_distances.argmax())
        return candidates[selected.cpu()].contiguous()

    @staticmethod
    def _batch_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
        if "image" not in batch:
            raise KeyError("Every training batch must contain 'image'")
        labels = batch.get("label")
        if labels is not None and torch.any(torch.as_tensor(labels) != 0):
            raise ValueError("PatchCore.fit accepts only clean training images")
        return batch["image"].to(device, non_blocking=True)

    @torch.no_grad()
    def fit(
        self,
        train_loader: Iterable[Mapping[str, Any]],
        *,
        device: torch.device | str,
        validation_loader: Iterable[Mapping[str, Any]] | None = None,
        work_dir: str | None = None,
        resume: bool = False,
    ) -> PatchCore:
        """Build the bounded coreset memory bank and clean-score calibration."""
        device = torch.device(device)
        self.to(device)
        self.train(False)
        self.memory_bank = torch.empty(0, 0, device=device)
        self.image_score_scale.zero_()
        self.anomaly_map_scale.zero_()
        self.fit_summary = {}
        generator = torch.Generator().manual_seed(self.sampling_seed)
        embedding_chunks: list[torch.Tensor] = []
        priority_chunks: list[torch.Tensor] = []
        buffered_count = 0
        num_images = 0
        sampled_embeddings = 0
        patch_grid: tuple[int, int] | None = None

        for batch in tqdm(train_loader, desc="PatchCore feature extraction"):
            images = self._batch_images(batch, device)
            embeddings, patch_grid = self._patch_embeddings(images)
            sampled = self._sample_image_patches(embeddings, generator).cpu()
            priorities = torch.rand(sampled.shape[0], generator=generator)
            embedding_chunks.append(sampled)
            priority_chunks.append(priorities)
            buffered_count += sampled.shape[0]
            sampled_embeddings += sampled.shape[0]
            num_images += images.shape[0]
            if buffered_count >= 2 * self.max_training_embeddings:
                embedding_chunks, priority_chunks, buffered_count = (
                    self._reduce_priority_reservoir(
                        embedding_chunks,
                        priority_chunks,
                        self.max_training_embeddings,
                    )
                )

        if not embedding_chunks or patch_grid is None:
            raise ValueError("train_loader produced no images")
        embedding_chunks, _, candidate_count = self._reduce_priority_reservoir(
            embedding_chunks,
            priority_chunks,
            self.max_training_embeddings,
        )
        candidates = embedding_chunks[0]
        self.memory_bank = self._build_coreset(
            candidates,
            device=device,
            generator=generator,
        ).to(device)

        image_scores: list[torch.Tensor] = []
        patch_scores: list[torch.Tensor] = []
        for batch_index, batch in enumerate(
            tqdm(train_loader, desc="PatchCore calibration", leave=False)
        ):
            if batch_index >= self.calibration_batches:
                break
            images = self._batch_images(batch, device)
            raw_image_scores, raw_patch_scores, _ = self._raw_scores(images)
            image_scores.append(raw_image_scores.cpu())
            patch_scores.append(raw_patch_scores.flatten().cpu())
        if not image_scores:
            raise ValueError("train_loader must be re-iterable for calibration")

        epsilon = torch.finfo(torch.float32).eps
        image_scale = torch.quantile(
            torch.cat(image_scores).float(), self.calibration_quantile
        ).clamp_min(epsilon)
        map_scale = torch.quantile(
            torch.cat(patch_scores).float(), self.calibration_quantile
        ).clamp_min(epsilon)
        self.image_score_scale.copy_(image_scale.to(device))
        self.anomaly_map_scale.copy_(map_scale.to(device))
        self.fit_summary = {
            "num_training_images": num_images,
            "sampled_embeddings": sampled_embeddings,
            "reservoir_embeddings": candidate_count,
            "memory_bank_embeddings": self.memory_bank.shape[0],
            "embedding_dimension": self.memory_bank.shape[1],
            "patch_grid": list(patch_grid),
            "image_score_scale": float(self.image_score_scale),
            "anomaly_map_scale": float(self.anomaly_map_scale),
        }
        return self

    def _nearest_memory(
        self,
        queries: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if self.memory_bank.numel() == 0:
            raise RuntimeError("PatchCore must be fitted before nearest-neighbour search")
        if queries.shape[1] != self.memory_bank.shape[1]:
            raise ValueError("Query and memory-bank embedding dimensions do not match")

        nearest_distances = torch.empty(queries.shape[0], device=queries.device)
        nearest_indices = torch.empty(
            queries.shape[0], dtype=torch.long, device=queries.device
        )
        for query_start in range(0, queries.shape[0], self.distance_query_chunk_size):
            query_end = min(query_start + self.distance_query_chunk_size, queries.shape[0])
            query_chunk = queries[query_start:query_end]
            best_distances = torch.full(
                (query_chunk.shape[0],), torch.inf, device=queries.device
            )
            best_indices = torch.zeros(
                query_chunk.shape[0], dtype=torch.long, device=queries.device
            )
            for bank_start in range(0, self.memory_bank.shape[0], self.distance_bank_chunk_size):
                bank_end = min(
                    bank_start + self.distance_bank_chunk_size,
                    self.memory_bank.shape[0],
                )
                distances = torch.cdist(query_chunk, self.memory_bank[bank_start:bank_end])
                block_distances, block_indices = distances.min(dim=1)
                improved = block_distances < best_distances
                best_distances[improved] = block_distances[improved]
                best_indices[improved] = block_indices[improved] + bank_start
            nearest_distances[query_start:query_end] = best_distances
            nearest_indices[query_start:query_end] = best_indices
        return nearest_distances, nearest_indices

    def _reweighted_image_scores(
        self,
        embeddings: torch.Tensor,
        patch_scores: torch.Tensor,
        nearest_indices: torch.Tensor,
    ) -> torch.Tensor:
        maximum_scores, maximum_locations = patch_scores.max(dim=1)
        if self.num_neighbors == 1 or self.memory_bank.shape[0] == 1:
            return maximum_scores

        scores = []
        num_support = min(self.num_neighbors, self.memory_bank.shape[0])
        for batch_index in range(embeddings.shape[0]):
            patch_index = maximum_locations[batch_index]
            query = embeddings[batch_index, patch_index]
            anchor_index = nearest_indices[batch_index, patch_index]
            anchor = self.memory_bank[anchor_index]
            anchor_distances = torch.linalg.vector_norm(self.memory_bank - anchor, dim=1)
            support_indices = anchor_distances.topk(
                num_support, largest=False
            ).indices
            support_distances = torch.linalg.vector_norm(
                self.memory_bank[support_indices] - query,
                dim=1,
            )
            anchor_position = (support_indices == anchor_index).nonzero(as_tuple=False)[0, 0]
            weight = 1.0 - torch.softmax(support_distances, dim=0)[anchor_position]
            scores.append(maximum_scores[batch_index] * weight)
        return torch.stack(scores)

    def _raw_scores(
        self,
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, tuple[int, int]]:
        embeddings, patch_grid = self._patch_embeddings(images)
        batch_size, num_patches, embedding_dim = embeddings.shape
        distances, nearest_indices = self._nearest_memory(
            embeddings.reshape(-1, embedding_dim)
        )
        patch_scores = distances.reshape(batch_size, num_patches)
        nearest_indices = nearest_indices.reshape(batch_size, num_patches)
        image_scores = self._reweighted_image_scores(
            embeddings, patch_scores, nearest_indices
        )
        return image_scores, patch_scores, patch_grid

    @staticmethod
    def _normalize(raw_scores: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        epsilon = torch.finfo(raw_scores.dtype).eps
        return raw_scores / (raw_scores + scale.clamp_min(epsilon))

    @torch.no_grad()
    def predict_with_raw(
        self,
        images: torch.Tensor,
    ) -> tuple[AnomalyPrediction, torch.Tensor, torch.Tensor]:
        if not self.is_fitted:
            raise RuntimeError("PatchCore must be fitted before predict")
        input_size = images.shape[-2:]
        raw_image_scores, raw_patch_scores, patch_grid = self._raw_scores(images)
        raw_map = raw_patch_scores.reshape(images.shape[0], 1, *patch_grid)
        raw_map = F.interpolate(
            raw_map,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )
        if self.gaussian_sigma > 0:
            kernel_size = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            raw_map = gaussian_blur(
                raw_map,
                [kernel_size, kernel_size],
                [self.gaussian_sigma, self.gaussian_sigma],
            )
        prediction = AnomalyPrediction(
            anomaly_score=self._normalize(raw_image_scores, self.image_score_scale),
            anomaly_map=self._normalize(raw_map, self.anomaly_map_scale),
        )
        return prediction, raw_image_scores, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        prediction, _, _ = self.predict_with_raw(images)
        return prediction

    def _prepare_state_dict_for_load(self, state_dict: Mapping[str, torch.Tensor]) -> None:
        memory_bank = state_dict.get("memory_bank")
        if memory_bank is not None and memory_bank.shape != self.memory_bank.shape:
            self.memory_bank = torch.empty_like(memory_bank)

def build_model():
    if MODEL_NAME == "patchcore":
        return PatchCore(
            backbone=PATCHCORE_BACKBONE, pretrained=PATCHCORE_PRETRAINED,
            layers=PATCHCORE_LAYERS,
            coreset_sampling_ratio=PATCHCORE_CORESET_SAMPLING_RATIO,
            num_neighbors=PATCHCORE_NUM_NEIGHBORS,
            patch_size=PATCHCORE_PATCH_SIZE, patch_stride=PATCHCORE_PATCH_STRIDE,
            pretrain_embed_dimension=PATCHCORE_PRETRAIN_EMBED_DIMENSION,
            target_embed_dimension=PATCHCORE_TARGET_EMBED_DIMENSION,
            max_patches_per_image=PATCHCORE_MAX_PATCHES_PER_IMAGE,
            max_training_embeddings=PATCHCORE_MAX_TRAINING_EMBEDDINGS,
            max_memory_bank_size=PATCHCORE_MAX_MEMORY_BANK_SIZE,
            projection_dim=PATCHCORE_PROJECTION_DIM,
            sampling_seed=PATCHCORE_SAMPLING_SEED,
            calibration_quantile=PATCHCORE_CALIBRATION_QUANTILE,
            calibration_batches=PATCHCORE_CALIBRATION_BATCHES,
            distance_query_chunk_size=PATCHCORE_DISTANCE_QUERY_CHUNK_SIZE,
            distance_bank_chunk_size=PATCHCORE_DISTANCE_BANK_CHUNK_SIZE,
            gaussian_sigma=PATCHCORE_GAUSSIAN_SIGMA,
        )
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

def build_optimizer(model):
    if OPTIMIZER_NAME == "none":
        if any(parameter.requires_grad for parameter in model.parameters()):
            raise ValueError("optimizer=none requires a fully frozen model")
        return None
    raise ValueError("PatchCore requires OPTIMIZER_NAME='none'")

def build_scheduler(optimizer):
    if SCHEDULER_NAME == "none":
        return None
    if optimizer is None:
        raise ValueError("A scheduler requires an optimizer")
    raise ValueError("PatchCore requires SCHEDULER_NAME='none'")


In [ ]:
from __future__ import annotations

ESSENTIAL_METRIC_NAMES = (
    "image_auroc",
    "image_average_precision",
    "pixel_auroc",
    "pixel_average_precision",
)


class ExactBinaryMetrics:
    """Exact binary AUROC and average precision for small score collections."""

    def __init__(self) -> None:
        self._scores: list[torch.Tensor] = []
        self._targets: list[torch.Tensor] = []

    @torch.no_grad()
    def update(self, scores: torch.Tensor, targets: torch.Tensor) -> None:
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        self._scores.append(scores.clone())
        self._targets.append(targets.clone())

    def compute(self) -> dict[str, float]:
        if not self._scores:
            raise ValueError("AUROC and average precision require both target classes")
        scores = torch.cat(self._scores)
        targets = torch.cat(self._targets)
        positives = int(targets.sum())
        negatives = targets.numel() - positives
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")

        order = torch.argsort(scores, descending=True)
        sorted_scores = scores[order]
        sorted_targets = targets[order]
        _, group_counts = torch.unique_consecutive(sorted_scores, return_counts=True)
        group_ends = group_counts.cumsum(0) - 1
        true_positives = sorted_targets.cumsum(0)[group_ends].double()
        false_positives = (~sorted_targets).cumsum(0)[group_ends].double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives)

        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)),
            torch.cat((zero, false_positive_rate)),
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {
            "auroc": float(auroc),
            "average_precision": float(average_precision),
        }


class BinaryHistogramMetrics:
    """Memory-bounded binary AUROC and average precision accumulator."""

    def __init__(self, num_bins: int = 2048) -> None:
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        self.num_bins = int(num_bins)
        self.positive_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.negative_histogram = torch.zeros(num_bins, dtype=torch.int64)

    @torch.no_grad()
    def update(
        self,
        scores: torch.Tensor,
        targets: torch.Tensor,
        valid_mask: torch.Tensor | None = None,
    ) -> None:
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if valid_mask is not None:
            valid_mask = valid_mask.detach().reshape(-1).cpu().bool()
            if valid_mask.numel() != scores.numel():
                raise ValueError("valid_mask must match scores")
            scores = scores[valid_mask]
            targets = targets[valid_mask]
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("scores must be normalized to [0, 1]")

        bins = (scores * self.num_bins).long().clamp(max=self.num_bins - 1)
        self.positive_histogram += torch.bincount(
            bins[targets], minlength=self.num_bins
        )
        self.negative_histogram += torch.bincount(
            bins[~targets], minlength=self.num_bins
        )

    def compute(self) -> dict[str, float]:
        positives = int(self.positive_histogram.sum())
        negatives = int(self.negative_histogram.sum())
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")

        true_positives = self.positive_histogram.flip(0).cumsum(0).double()
        false_positives = self.negative_histogram.flip(0).cumsum(0).double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives).clamp_min(1)

        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)),
            torch.cat((zero, false_positive_rate)),
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {
            "auroc": float(auroc),
            "average_precision": float(average_precision),
        }


class AnomalyMetrics:
    """Accumulate primary, target-wheel, and score-saturation metrics."""

    def __init__(self, histogram_bins: int = 2048) -> None:
        self.image = ExactBinaryMetrics()
        self.pixel = BinaryHistogramMetrics(histogram_bins)
        self.target_wheel_pixel = BinaryHistogramMetrics(histogram_bins)
        self.num_images = 0
        self.image_scores_at_zero = 0
        self.image_scores_at_one = 0

    @torch.no_grad()
    def update(
        self,
        prediction: AnomalyPrediction,
        labels: torch.Tensor,
        anomaly_masks: torch.Tensor,
        *,
        image_scores: torch.Tensor | None = None,
        valid_pixel_mask: torch.Tensor | None = None,
        target_wheel_mask: torch.Tensor | None = None,
    ) -> None:
        labels = labels.reshape(-1)
        if labels.shape != prediction.anomaly_score.shape:
            raise ValueError("labels must match anomaly_score shape")
        metric_image_scores = (
            prediction.anomaly_score if image_scores is None else image_scores
        ).reshape(-1)
        if metric_image_scores.shape != labels.shape:
            raise ValueError("image_scores must match labels")

        anomaly_masks = anomaly_masks > 0
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if anomaly_masks.shape != prediction.anomaly_map.shape:
            raise ValueError("anomaly_masks must match anomaly_map shape")

        def prepare_mask(mask: torch.Tensor | None, name: str) -> torch.Tensor | None:
            if mask is None:
                return None
            mask = mask > 0
            if mask.ndim == 3:
                mask = mask.unsqueeze(1)
            if mask.shape != prediction.anomaly_map.shape:
                raise ValueError(f"{name} must match anomaly_map shape")
            return mask

        valid_pixel_mask = prepare_mask(valid_pixel_mask, "valid_pixel_mask")
        target_wheel_mask = prepare_mask(target_wheel_mask, "target_wheel_mask")
        if target_wheel_mask is None:
            target_wheel_mask = torch.ones_like(anomaly_masks, dtype=torch.bool)

        normalized_scores = prediction.anomaly_score.detach()
        self.num_images += normalized_scores.numel()
        self.image_scores_at_zero += int((normalized_scores == 0).sum())
        self.image_scores_at_one += int((normalized_scores == 1).sum())

        self.image.update(metric_image_scores, labels)
        self.pixel.update(
            prediction.anomaly_map,
            anomaly_masks,
            valid_mask=valid_pixel_mask,
        )
        self.target_wheel_pixel.update(
            prediction.anomaly_map,
            anomaly_masks,
            valid_mask=target_wheel_mask,
        )

    def compute(self) -> dict[str, float | int]:
        image = self.image.compute()
        pixel = self.pixel.compute()
        target_wheel_pixel = self.target_wheel_pixel.compute()
        saturated = self.image_scores_at_zero + self.image_scores_at_one
        return {
            "image_auroc": image["auroc"],
            "image_average_precision": image["average_precision"],
            "pixel_auroc": pixel["auroc"],
            "pixel_average_precision": pixel["average_precision"],
            "target_wheel_pixel_auroc": target_wheel_pixel["auroc"],
            "target_wheel_pixel_average_precision": (
                target_wheel_pixel["average_precision"]
            ),
            "normalized_image_scores_at_zero": self.image_scores_at_zero,
            "normalized_image_scores_at_one": self.image_scores_at_one,
            "normalized_image_score_saturation_fraction": (
                saturated / self.num_images if self.num_images else 0.0
            ),
        }



def update_metrics_from_batch(
    metrics: AnomalyMetrics,
    prediction: AnomalyPrediction,
    batch: dict[str, torch.Tensor],
    *,
    image_scores: torch.Tensor | None = None,
    restrict_pixels_to_target_mask: bool = False,
) -> None:
    """Update full-frame metrics plus the additional target-wheel view."""
    valid_pixel_mask = batch["target_mask"] if restrict_pixels_to_target_mask else None
    metrics.update(
        prediction,
        batch["label"],
        batch["anomaly_mask"],
        image_scores=image_scores,
        valid_pixel_mask=valid_pixel_mask,
        target_wheel_mask=batch["target_mask"],
    )


@torch.no_grad()
def evaluate_anomaly_detector(
    detector: AnomalyDetector,
    loader: Iterable[Mapping[str, Any]],
    *,
    device: torch.device | str,
    histogram_bins: int = 2048,
    restrict_pixels_to_target_mask: bool = False,
    description: str = "Evaluation",
    diagnostics: AnomalyDiagnostics | None = None,
) -> dict[str, float | int]:
    """Evaluate one fitted detector and optionally collect diagnostics."""
    if not detector.is_fitted:
        raise RuntimeError("The anomaly detector must be fitted before evaluation")
    device = torch.device(device)
    detector.to(device)
    detector.train(False)
    metrics = AnomalyMetrics(histogram_bins=histogram_bins)
    num_images = 0

    for batch in tqdm(loader, desc=description):
        images = batch["image"].to(device, non_blocking=True)
        predict_with_raw = getattr(detector, "predict_with_raw", None)
        if predict_with_raw is None:
            prediction = detector.predict(images)
            raw_image_scores = prediction.anomaly_score
        else:
            prediction, raw_image_scores, _ = predict_with_raw(images)
        update_metrics_from_batch(
            metrics,
            prediction,
            batch,
            image_scores=raw_image_scores,
            restrict_pixels_to_target_mask=restrict_pixels_to_target_mask,
        )
        if diagnostics is not None:
            diagnostics.update(prediction, raw_image_scores, batch)
        num_images += images.shape[0]
    if num_images == 0:
        raise ValueError("Evaluation loader produced no images")
    return metrics.compute()


In [ ]:
from __future__ import annotations

def prepare_data(dataset_root):
    evaluation_config = PreprocessingConfig(
        resize=PREPROCESS_RESIZE,
        resize_shorter_side=PREPROCESS_RESIZE_SHORTER_SIDE,
        center_crop=PREPROCESS_CENTER_CROP,
        pose_crop_enabled=POSE_CROP_ENABLED,
        pose_crops=POSE_CROPS,
        normalize_mean=NORMALIZE_MEAN,
        normalize_std=NORMALIZE_STD,
        augmentations_enabled=False,
    )
    # PatchCore uses the same deterministic geometry for all splits.
    if TRAIN_AUGMENTATIONS_ENABLED:
        raise ValueError("This Reference comparison requires augmentation-free fitting")
    train_config = evaluation_config
    train_preprocessing = WheelPreprocessor(train_config)
    evaluation_preprocessing = WheelPreprocessor(evaluation_config)
    loaders = tuple(
        (
            split,
            build_dataloader(
                dataset_root, split,
                batch_size=BATCH_SIZE if split == "train" else EVALUATION_BATCH_SIZE,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
                seed=SEED,
                preprocessing=train_preprocessing if split == "train" else evaluation_preprocessing,
                camera_poses=CAMERA_POSES,
            ),
        )
        for split in ("train", "validation", "test")
    )
    # Each Dataset validates the complete manifest before applying its pose filter.
    observed_counts = Counter(
        (row["split"], row["condition"])
        for _, loader in loaders for row in loader.dataset.rows
    )
    if observed_counts != EXPECTED_SPLIT_CONDITION_COUNTS:
        raise ValueError(
            f"Unexpected pose-A counts: {dict(observed_counts)}; "
            f"expected {dict(EXPECTED_SPLIT_CONDITION_COUNTS)}"
        )
    if any(
        row["camera_pose"] not in CAMERA_POSES
        for _, loader in loaders for row in loader.dataset.rows
    ):
        raise ValueError("A sample outside the requested camera poses was selected")
    if any(row["condition"] != "clean" for row in loaders[0][1].dataset.rows):
        raise ValueError("PatchCore training must contain only clean images")
    return {
        "train_config": train_config,
        "train_preprocessing": train_preprocessing,
        "evaluation_config": evaluation_config,
        "evaluation_preprocessing": evaluation_preprocessing,
        "train_loader": loaders[0][1],
        "validation_loader": loaders[1][1],
        "test_loader": loaders[2][1],
        "loaders": loaders,
    }


def audit_processed_hole_visibility(loaders):
    """Inspect preprocessing geometry without fitting or excluding any samples."""
    report = {
        "camera_poses": list(CAMERA_POSES),
        "pose_crops": POSE_CROPS,
        "input_size": list(IMAGE_SIZE),
        "samples_removed": 0,
        "holes": [],
        "empty_processed_holes": [],
        "partially_clipped_holes": [],
    }
    for split, loader in loaders:
        dataset = loader.dataset
        config = dataset.preprocessing.config
        for index, row in enumerate(tqdm(dataset.rows, desc=f"{split} mask audit")):
            if row["condition"] != "hole":
                continue
            sample = dataset[index]
            raw = pil_to_tensor(dataset._load_image(row["anomaly_mask_path"], "L"))
            pose_mask = raw
            if config.pose_crop_enabled:
                crop = config.pose_crops[row["camera_pose"]]
                if crop is not None:
                    top, left, height, width = crop
                    top = min(max(top, 0), raw.shape[-2] - height)
                    left = min(max(left, 0), raw.shape[-1] - width)
                    pose_mask = TF.crop(raw, top, left, height, width)
            resized = pose_mask
            if config.resize_shorter_side is not None:
                resized = TF.resize(
                    resized, config.resize_shorter_side,
                    interpolation=InterpolationMode.NEAREST,
                )
            elif config.resize is not None:
                resized = TF.resize(
                    resized, config.resize,
                    interpolation=InterpolationMode.NEAREST,
                )
            raw_pixels = int((raw > 0).sum())
            crop_pixels = int((pose_mask > 0).sum())
            resized_pixels = int((resized > 0).sum())
            processed_pixels = int((sample["anomaly_mask"] > 0).sum())
            record = {
                "split": split,
                "image_id": row["image_id"],
                "raw_hole_pixels": raw_pixels,
                "pose_crop_hole_pixels": crop_pixels,
                "resized_hole_pixels_before_center_crop": resized_pixels,
                "processed_hole_pixels": processed_pixels,
                "center_crop_retention_fraction": (
                    processed_pixels / resized_pixels if resized_pixels else 0.0
                ),
                "processed_target_wheel_pixels": int((sample["target_mask"] > 0).sum()),
            }
            report["holes"].append(record)
            if not processed_pixels or not record["processed_target_wheel_pixels"]:
                report["empty_processed_holes"].append(row["image_id"])
            if crop_pixels < raw_pixels or processed_pixels < resized_pixels:
                report["partially_clipped_holes"].append(row["image_id"])
    report["hole_count"] = len(report["holes"])
    report["minimum_processed_hole_pixels"] = min(
        (record["processed_hole_pixels"] for record in report["holes"]),
        default=None,
    )
    return report


In [ ]:
from __future__ import annotations

def audit_preprocessing(
    root: str | Path,
    config: PreprocessingConfig,
    *,
    split: str = "train",
    indices: Sequence[int] | None = None,
    camera_poses: Sequence[str] | None = None,
    variants: int = 2,
    seed: int = 42,
) -> plt.Figure:
    """Plot raw samples beside reproducible stochastic preprocessing variants."""
    if variants < 1:
        raise ValueError("variants must be at least 1")
    dataset = CuriosityWheelDataset(root, split, camera_poses=camera_poses)
    selected = list(indices) if indices is not None else list(range(min(4, len(dataset))))
    if not selected:
        raise ValueError("indices must select at least one sample")
    if any(index < 0 or index >= len(dataset) for index in selected):
        raise IndexError("audit sample index is outside the dataset")

    preprocessor = WheelPreprocessor(config)
    raw_preprocessor = WheelPreprocessor()
    figure, axes = plt.subplots(
        len(selected), variants + 1,
        figsize=(4.5 * (variants + 1), 3.5 * len(selected)),
        squeeze=False,
    )
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        for row_index, sample_index in enumerate(selected):
            sample = dataset[sample_index]
            raw_image = sample["image"]
            target_mask = sample["target_mask"]
            anomaly_mask = sample["anomaly_mask"]
            image_id = sample["metadata"]["image_id"]
            axes[row_index, 0].imshow(
                raw_preprocessor.image_for_display(raw_image).permute(1, 2, 0)
            )
            axes[row_index, 0].set_title(f"Original\n{image_id}")
            for variant in range(variants):
                processed, _, _ = preprocessor(
                    raw_image.clone(),
                    target_mask.clone(),
                    anomaly_mask.clone(),
                    camera_pose=sample["metadata"].get("camera_pose"),
                )
                axes[row_index, variant + 1].imshow(
                    preprocessor.image_for_display(processed).permute(1, 2, 0)
                )
                axes[row_index, variant + 1].set_title(f"Processed {variant + 1}")
            for axis in axes[row_index]:
                axis.axis("off")
    figure.suptitle(f"Audit preprocessing — split {split}")
    figure.tight_layout()
    return figure


In [ ]:
from __future__ import annotations

def smoke_test_loaders(loaders):
    # Verify the common numerical contract on one batch per split.
    for split, loader in loaders:
        batch = next(iter(loader))
        images = batch["image"]
        if images.dtype != torch.float32 or not torch.isfinite(images).all():
            raise TypeError(f"{split} images must be finite float32 tensors")
        if (
            batch["target_mask"].dtype != torch.uint8
            or batch["anomaly_mask"].dtype != torch.uint8
        ):
            raise TypeError(f"{split} masks must remain uint8 tensors")
        print(
            f"{split:10s} image={tuple(images.shape)} {images.dtype} "
            f"range=[{images.min().item():.3f}, {images.max().item():.3f}] "
            f"target_mask={tuple(batch['target_mask'].shape)} "
            f"anomaly_mask={tuple(batch['anomaly_mask'].shape)} "
            f"label={batch['label'].tolist()}"
        )

In [ ]:
from __future__ import annotations

@torch.no_grad()
def collect_anomaly_visualization_samples(
    detector: AnomalyDetector,
    loader: Iterable[Mapping[str, Any]],
    *,
    device: torch.device | str,
    preprocessor: WheelPreprocessor,
    num_clean: int = 3,
    num_anomalous: int = 3,
) -> list[dict[str, Any]]:
    """Collect a small, deterministic set of clean and anomalous predictions."""
    if num_clean < 0 or num_anomalous < 0 or num_clean + num_anomalous == 0:
        raise ValueError("At least one non-negative visualization quota is required")
    if not detector.is_fitted:
        raise RuntimeError("The anomaly detector must be fitted before visualization")

    device = torch.device(device)
    detector.to(device)
    detector.train(False)
    remaining = {0: int(num_clean), 1: int(num_anomalous)}
    samples: list[dict[str, Any]] = []

    for batch in loader:
        labels = torch.as_tensor(batch["label"]).reshape(-1)
        selected = [
            index
            for index, label in enumerate(labels.tolist())
            if label in remaining and remaining[label] > 0
        ]
        if not selected:
            continue

        images = batch["image"][selected].to(device, non_blocking=True)
        prediction = detector.predict(images)
        metadata = batch.get("metadata", {})
        image_ids = metadata.get("image_id", [None] * labels.numel())
        for prediction_index, batch_index in enumerate(selected):
            label = int(labels[batch_index])
            if remaining[label] == 0:
                continue
            samples.append(
                {
                    "image": preprocessor.image_for_display(
                        batch["image"][batch_index]
                    ).cpu(),
                    "target_mask": batch["target_mask"][batch_index].cpu(),
                    "anomaly_mask": batch["anomaly_mask"][batch_index].cpu(),
                    "anomaly_map": prediction.anomaly_map[prediction_index].cpu(),
                    "anomaly_score": float(
                        prediction.anomaly_score[prediction_index].cpu()
                    ),
                    "label": label,
                    "image_id": image_ids[batch_index],
                }
            )
            remaining[label] -= 1
        if not any(remaining.values()):
            break

    if any(remaining.values()):
        raise ValueError(
            "The loader does not contain enough samples for the requested quotas: "
            f"missing clean={remaining[0]}, anomalous={remaining[1]}"
        )
    return samples


def plot_anomaly_visualizations(
    samples: list[dict[str, Any]],
    *,
    title: str,
    colormap: str = "magma",
    overlay_alpha: float = 0.55,
) -> Figure:
    """Plot input, ground truth, normalized anomaly map, and image overlay."""
    if not samples:
        raise ValueError("samples cannot be empty")
    if not 0 <= overlay_alpha <= 1:
        raise ValueError("overlay_alpha must be in [0, 1]")

    figure = Figure(figsize=(16, 3.7 * len(samples)), layout="constrained")
    FigureCanvasAgg(figure)
    axes = figure.subplots(len(samples), 4, squeeze=False)
    heatmap_artist = None
    for row, sample in enumerate(samples):
        image = sample["image"].permute(1, 2, 0).numpy()
        target_mask = sample["target_mask"].squeeze().numpy() > 0
        anomaly_mask = sample["anomaly_mask"].squeeze().numpy() > 0
        anomaly_map = sample["anomaly_map"].squeeze().numpy()
        label = "anomalous" if sample["label"] else "clean"
        image_id = sample.get("image_id") or "unknown"
        raw_score = sample.get("raw_anomaly_score")
        score_text = f"score={sample['anomaly_score']:.3f}"
        if raw_score is not None:
            score_text += f" | raw={raw_score:.3f}"

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(
            f"Input — {label}\n{score_text} | {image_id}"
        )

        axes[row, 1].imshow(anomaly_mask, cmap="gray", vmin=0, vmax=1)
        if target_mask.any() and not target_mask.all():
            axes[row, 1].contour(target_mask, levels=[0.5], colors="cyan", linewidths=1)
        axes[row, 1].set_title("Ground truth\nwhite=hole, cyan=wheel")

        heatmap_artist = axes[row, 2].imshow(
            anomaly_map, cmap=colormap, vmin=0, vmax=1
        )
        if target_mask.any() and not target_mask.all():
            axes[row, 2].contour(target_mask, levels=[0.5], colors="cyan", linewidths=1)
        axes[row, 2].set_title("Anomaly map\nnormalized score [0, 1]")

        axes[row, 3].imshow(image)
        axes[row, 3].imshow(
            anomaly_map,
            cmap=colormap,
            vmin=0,
            vmax=1,
            alpha=overlay_alpha,
        )
        if anomaly_mask.any() and not anomaly_mask.all():
            axes[row, 3].contour(
                anomaly_mask, levels=[0.5], colors="lime", linewidths=1.5
            )
        axes[row, 3].set_title("Overlay\ngreen=ground-truth boundary")

        for axis in axes[row]:
            axis.axis("off")

    figure.suptitle(title, fontsize=15)
    figure.colorbar(
        heatmap_artist,
        ax=axes[:, 2:].ravel().tolist(),
        label="Normalized anomaly score",
        shrink=0.8,
    )
    return figure


def save_anomaly_visualizations(
    detector: AnomalyDetector,
    loader: Iterable[Mapping[str, Any]],
    output_path: str | Path,
    *,
    device: torch.device | str,
    preprocessor: WheelPreprocessor,
    title: str,
    num_clean: int = 3,
    num_anomalous: int = 3,
    colormap: str = "magma",
    overlay_alpha: float = 0.55,
    dpi: int = 150,
) -> Path:
    """Collect representative predictions and save one comparison figure."""
    if dpi < 1:
        raise ValueError("dpi must be at least 1")
    samples = collect_anomaly_visualization_samples(
        detector,
        loader,
        device=device,
        preprocessor=preprocessor,
        num_clean=num_clean,
        num_anomalous=num_anomalous,
    )
    figure = plot_anomaly_visualizations(
        samples,
        title=title,
        colormap=colormap,
        overlay_alpha=overlay_alpha,
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=dpi, bbox_inches="tight")
    figure.clear()
    return output_path


In [ ]:
from __future__ import annotations

DEFAULT_GROUP_FIELDS = ("severity", "camera_pose", "lighting", "wear")


def _score_statistics(values: list[float]) -> dict[str, float | int]:
    if not values:
        return {"count": 0}
    scores = torch.tensor(values, dtype=torch.float64)
    return {
        "count": scores.numel(),
        "minimum": float(scores.min()),
        "q25": float(torch.quantile(scores, 0.25)),
        "median": float(torch.quantile(scores, 0.50)),
        "q75": float(torch.quantile(scores, 0.75)),
        "maximum": float(scores.max()),
        "mean": float(scores.mean()),
        "std": float(scores.std(unbiased=False)),
    }


def _connected_components(mask: torch.Tensor) -> list[list[int]]:
    """Return 4-connected positive regions as flattened pixel indices."""
    mask = mask.detach().cpu().bool().squeeze()
    if mask.ndim != 2:
        raise ValueError("PRO masks must be two-dimensional after squeezing")
    height, width = mask.shape
    remaining = set(torch.nonzero(mask.flatten(), as_tuple=False).flatten().tolist())
    components: list[list[int]] = []
    while remaining:
        start = remaining.pop()
        queue = deque([start])
        component = [start]
        while queue:
            index = queue.popleft()
            row, column = divmod(index, width)
            neighbours = []
            if row > 0:
                neighbours.append(index - width)
            if row + 1 < height:
                neighbours.append(index + width)
            if column > 0:
                neighbours.append(index - 1)
            if column + 1 < width:
                neighbours.append(index + 1)
            for neighbour in neighbours:
                if neighbour in remaining:
                    remaining.remove(neighbour)
                    queue.append(neighbour)
                    component.append(neighbour)
        components.append(component)
    return components


class PerRegionOverlap:
    """Memory-bounded PRO curve and normalized AUPRO accumulator."""

    def __init__(self, num_bins: int = 256, max_fpr: float = 0.30) -> None:
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        if not 0 < max_fpr <= 1:
            raise ValueError("max_fpr must be in (0, 1]")
        self.num_bins = int(num_bins)
        self.max_fpr = float(max_fpr)
        self.normal_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.region_overlap_sum = torch.zeros(num_bins, dtype=torch.float64)
        self.num_regions = 0

    @torch.no_grad()
    def update(
        self,
        scores: torch.Tensor,
        anomaly_masks: torch.Tensor,
        valid_mask: torch.Tensor | None = None,
    ) -> None:
        scores = scores.detach().cpu()
        anomaly_masks = anomaly_masks.detach().cpu() > 0
        if scores.ndim == 3:
            scores = scores.unsqueeze(1)
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if scores.shape != anomaly_masks.shape:
            raise ValueError("scores and anomaly_masks must have matching shapes")
        if valid_mask is None:
            valid_mask = torch.ones_like(anomaly_masks, dtype=torch.bool)
        else:
            valid_mask = valid_mask.detach().cpu() > 0
            if valid_mask.ndim == 3:
                valid_mask = valid_mask.unsqueeze(1)
            if valid_mask.shape != scores.shape:
                raise ValueError("valid_mask must match scores")

        for score, anomaly_mask, valid in zip(scores, anomaly_masks, valid_mask):
            score = score.squeeze(0)
            anomaly_mask = anomaly_mask.squeeze(0) & valid.squeeze(0)
            valid = valid.squeeze(0)
            normal_scores = score[valid & ~anomaly_mask]
            if normal_scores.numel():
                bins = (normal_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                self.normal_histogram += torch.bincount(
                    bins, minlength=self.num_bins
                )
            flattened_scores = score.flatten()
            for component in _connected_components(anomaly_mask):
                component_scores = flattened_scores[component]
                bins = (component_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                histogram = torch.bincount(bins, minlength=self.num_bins)
                self.region_overlap_sum += (
                    histogram.flip(0).cumsum(0).double() / len(component)
                )
                self.num_regions += 1

    @staticmethod
    def _normalized_area(
        fpr: torch.Tensor,
        pro: torch.Tensor,
        max_fpr: float,
    ) -> float:
        below = fpr <= max_fpr
        clipped_fpr = fpr[below]
        clipped_pro = pro[below]
        if clipped_fpr[-1] < max_fpr:
            upper_index = int(torch.nonzero(fpr > max_fpr)[0])
            lower_index = upper_index - 1
            denominator = fpr[upper_index] - fpr[lower_index]
            weight = (
                0.0
                if denominator == 0
                else float((max_fpr - fpr[lower_index]) / denominator)
            )
            interpolated = pro[lower_index] + weight * (
                pro[upper_index] - pro[lower_index]
            )
            clipped_fpr = torch.cat(
                (clipped_fpr, torch.tensor([max_fpr], dtype=torch.float64))
            )
            clipped_pro = torch.cat((clipped_pro, interpolated.reshape(1)))
        area = torch.trapezoid(clipped_pro, clipped_fpr) / max_fpr
        return float(area)

    def compute(self) -> dict[str, Any]:
        total_normal = int(self.normal_histogram.sum())
        if total_normal == 0 or self.num_regions == 0:
            raise ValueError("PRO requires normal pixels and at least one anomaly region")
        fpr = self.normal_histogram.flip(0).cumsum(0).double() / total_normal
        pro = self.region_overlap_sum / self.num_regions
        fpr = torch.cat((torch.zeros(1, dtype=torch.float64), fpr))
        pro = torch.cat((torch.zeros(1, dtype=torch.float64), pro))

        reported_max_fprs = sorted({0.05, 0.10, self.max_fpr})
        aupro_by_max_fpr = {
            f"{max_fpr:.2f}": self._normalized_area(fpr, pro, max_fpr)
            for max_fpr in reported_max_fprs
        }
        return {
            "aupro": aupro_by_max_fpr[f"{self.max_fpr:.2f}"],
            "max_fpr": self.max_fpr,
            "aupro_by_max_fpr": aupro_by_max_fpr,
            "num_regions": self.num_regions,
            "curve": [
                {"false_positive_rate": float(x), "pro": float(y)}
                for x, y in zip(fpr, pro)
            ],
        }


class AnomalyDiagnostics:
    """Collect score distributions, grouped image/pixel metrics, and PRO."""

    def __init__(
        self,
        manifest_rows: Sequence[Mapping[str, str]],
        *,
        preprocessor: WheelPreprocessor,
        histogram_bins: int = 2048,
        pro_bins: int = 256,
        pro_max_fpr: float = 0.30,
        group_fields: Sequence[str] = DEFAULT_GROUP_FIELDS,
        num_extreme_examples: int = 6,
        restrict_pixels_to_target_mask: bool = False,
    ) -> None:
        if num_extreme_examples < 1:
            raise ValueError("num_extreme_examples must be at least 1")
        self.preprocessor = preprocessor
        self.histogram_bins = int(histogram_bins)
        self.group_fields = tuple(group_fields)
        self.num_extreme_examples = int(num_extreme_examples)
        self.restrict_pixels_to_target_mask = bool(restrict_pixels_to_target_mask)
        self.pro = PerRegionOverlap(pro_bins, pro_max_fpr)
        self.records: list[dict[str, Any]] = []
        self.group_metrics: dict[str, dict[str, BinaryHistogramMetrics]] = {
            field: {} for field in self.group_fields
        }
        self.group_image_counts: dict[str, dict[str, int]] = {
            field: {} for field in self.group_fields
        }
        self.top_clean: list[dict[str, Any]] = []
        self.bottom_hole: list[dict[str, Any]] = []
        self.anomalous_pixels = 0
        self.valid_pixels = 0
        self.pair_groups = {
            row.get("pair_id", ""): {
                field: row.get(field, "") for field in self.group_fields
            }
            for row in manifest_rows
            if row.get("condition") == "hole" and row.get("pair_id")
        }

    @staticmethod
    def _batch_value(metadata: Mapping[str, Any], key: str, index: int) -> Any:
        values = metadata.get(key)
        if values is None:
            return ""
        value = values[index]
        return value.item() if isinstance(value, torch.Tensor) else value

    def _group_value(
        self, metadata: Mapping[str, Any], field: str, index: int
    ) -> str:
        value = str(self._batch_value(metadata, field, index) or "")
        if value:
            return value
        pair_id = str(self._batch_value(metadata, "pair_id", index) or "")
        return str(self.pair_groups.get(pair_id, {}).get(field, ""))

    def _example(
        self,
        batch: Mapping[str, Any],
        prediction: AnomalyPrediction,
        raw_scores: torch.Tensor,
        index: int,
    ) -> dict[str, Any]:
        metadata = batch.get("metadata", {})
        return {
            "image": self.preprocessor.image_for_display(batch["image"][index]).cpu(),
            "target_mask": batch["target_mask"][index].cpu(),
            "anomaly_mask": batch["anomaly_mask"][index].cpu(),
            "anomaly_map": prediction.anomaly_map[index].detach().cpu(),
            "anomaly_score": float(prediction.anomaly_score[index].detach().cpu()),
            "raw_anomaly_score": float(raw_scores[index].detach().cpu()),
            "label": int(batch["label"][index]),
            "image_id": self._batch_value(metadata, "image_id", index),
        }

    @torch.no_grad()
    def update(
        self,
        prediction: AnomalyPrediction,
        raw_image_scores: torch.Tensor,
        batch: Mapping[str, Any],
    ) -> None:
        raw_image_scores = raw_image_scores.detach().reshape(-1).cpu()
        labels = torch.as_tensor(batch["label"]).reshape(-1).cpu()
        if raw_image_scores.shape != labels.shape:
            raise ValueError("raw_image_scores must match batch labels")
        anomaly_masks = batch["anomaly_mask"] > 0
        valid_mask = (
            batch["target_mask"] > 0
            if self.restrict_pixels_to_target_mask
            else torch.ones_like(anomaly_masks, dtype=torch.bool)
        )
        self.anomalous_pixels += int((anomaly_masks & valid_mask).sum())
        self.valid_pixels += int(valid_mask.sum())
        self.pro.update(prediction.anomaly_map, anomaly_masks, valid_mask)
        metadata = batch.get("metadata", {})

        for index, label_tensor in enumerate(labels):
            label = int(label_tensor)
            condition = "hole" if label else "clean"
            record = {
                "image_id": self._batch_value(metadata, "image_id", index),
                "pair_id": self._batch_value(metadata, "pair_id", index),
                "condition": condition,
                "raw_image_score": float(raw_image_scores[index]),
                "normalized_image_score": float(prediction.anomaly_score[index].cpu()),
            }
            for field in self.group_fields:
                value = self._group_value(metadata, field, index)
                record[field] = value
                if not value:
                    continue
                accumulator = self.group_metrics[field].setdefault(
                    value, BinaryHistogramMetrics(self.histogram_bins)
                )
                accumulator.update(
                    prediction.anomaly_map[index],
                    anomaly_masks[index],
                    valid_mask=valid_mask[index],
                )
                counts = self.group_image_counts[field]
                counts[value] = counts.get(value, 0) + 1
            self.records.append(record)

            example = self._example(batch, prediction, raw_image_scores, index)
            if label == 0:
                self.top_clean.append(example)
                self.top_clean.sort(
                    key=lambda item: item["raw_anomaly_score"], reverse=True
                )
                del self.top_clean[self.num_extreme_examples :]
            else:
                self.bottom_hole.append(example)
                self.bottom_hole.sort(key=lambda item: item["raw_anomaly_score"])
                del self.bottom_hole[self.num_extreme_examples :]

    def compute(self) -> dict[str, Any]:
        if not self.records or self.valid_pixels == 0:
            raise ValueError("Diagnostics require at least one evaluated image")
        grouped: dict[str, dict[str, Any]] = {}
        for field, values in self.group_metrics.items():
            grouped[field] = {}
            for value, metric in sorted(values.items()):
                positives = int(metric.positive_histogram.sum())
                negatives = int(metric.negative_histogram.sum())
                result = metric.compute() if positives and negatives else None
                grouped[field][value] = {
                    "pixel_average_precision": (
                        None if result is None else result["average_precision"]
                    ),
                    "anomalous_pixel_fraction": (
                        positives / (positives + negatives)
                        if positives + negatives
                        else 0.0
                    ),
                    "num_images": self.group_image_counts[field][value],
                }
        image_grouped: dict[str, dict[str, Any]] = {}
        for field in self.group_fields:
            image_grouped[field] = {}
            values = sorted(
                {str(row[field]) for row in self.records if row[field]}
            )
            for value in values:
                records = [row for row in self.records if row[field] == value]
                labels = torch.tensor(
                    [row["condition"] == "hole" for row in records],
                    dtype=torch.bool,
                )
                positives = int(labels.sum())
                negatives = labels.numel() - positives
                metrics = None
                if positives and negatives:
                    accumulator = ExactBinaryMetrics()
                    accumulator.update(
                        torch.tensor(
                            [row["raw_image_score"] for row in records],
                            dtype=torch.float32,
                        ),
                        labels,
                    )
                    metrics = accumulator.compute()
                image_grouped[field][value] = {
                    "image_auroc": None if metrics is None else metrics["auroc"],
                    "image_average_precision": (
                        None if metrics is None else metrics["average_precision"]
                    ),
                    "num_images": len(records),
                    "num_clean_images": negatives,
                    "num_anomalous_images": positives,
                }

        raw_scores = {
            condition: [
                row["raw_image_score"]
                for row in self.records
                if row["condition"] == condition
            ]
            for condition in ("clean", "hole")
        }
        return {
            "raw_image_score_distribution": {
                condition: _score_statistics(values)
                for condition, values in raw_scores.items()
            },
            "image_metrics_by_group": image_grouped,
            "pixel_average_precision_by_group": grouped,
            "anomalous_pixel_fraction": self.anomalous_pixels / self.valid_pixels,
            "random_pixel_ap_baseline": self.anomalous_pixels / self.valid_pixels,
            "pro": self.pro.compute(),
        }


def save_anomaly_diagnostics(
    diagnostics: AnomalyDiagnostics,
    output_dir: str | Path,
    *,
    split: str,
    colormap: str = "magma",
    overlay_alpha: float = 0.55,
    dpi: int = 150,
) -> tuple[dict[str, Any], list[Path]]:
    """Save diagnostic JSON, score/PRO CSVs, plots, and extreme examples."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    result = diagnostics.compute()
    json_result = {**result, "pro": {k: v for k, v in result["pro"].items() if k != "curve"}}
    summary_path = output_dir / f"{split}_diagnostics.json"
    summary_path.write_text(json.dumps(json_result, indent=2) + "\n", encoding="utf-8")

    score_path = output_dir / f"{split}_image_scores.csv"
    with score_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(diagnostics.records[0]))
        writer.writeheader()
        writer.writerows(diagnostics.records)

    pro_path = output_dir / f"{split}_pro_curve.csv"
    with pro_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["false_positive_rate", "pro"])
        writer.writeheader()
        writer.writerows(result["pro"]["curve"])

    score_figure = Figure(figsize=(8, 5), layout="constrained")
    FigureCanvasAgg(score_figure)
    axis = score_figure.subplots()
    for condition, color in (("clean", "tab:blue"), ("hole", "tab:orange")):
        values = [
            row["raw_image_score"]
            for row in diagnostics.records
            if row["condition"] == condition
        ]
        axis.hist(values, bins=50, density=True, alpha=0.55, label=condition, color=color)
    axis.set_title(f"{split.title()} raw image-score distribution")
    axis.set_xlabel("Raw PatchCore image score")
    axis.set_ylabel("Density")
    axis.legend()
    score_plot_path = output_dir / f"{split}_raw_score_distribution.png"
    score_figure.savefig(score_plot_path, dpi=dpi, bbox_inches="tight")
    score_figure.clear()

    paths = [summary_path, score_path, pro_path, score_plot_path]
    for name, examples, title in (
        ("highest_clean_scores", diagnostics.top_clean, "Clean images with highest raw scores"),
        ("lowest_hole_scores", diagnostics.bottom_hole, "Hole images with lowest raw scores"),
    ):
        if not examples:
            continue
        figure = plot_anomaly_visualizations(
            examples,
            title=f"{split.title()} — {title}",
            colormap=colormap,
            overlay_alpha=overlay_alpha,
        )
        path = output_dir / f"{split}_{name}.png"
        figure.savefig(path, dpi=dpi, bbox_inches="tight")
        figure.clear()
        paths.append(path)
    return json_result, paths


In [ ]:
from __future__ import annotations

# Save locally first, then optionally copy the same run artifacts to Drive.
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def save_json_atomic(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(data, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)
    return path

def get_kaggle_secret(name, required=False):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle secret: {name}")
    return value

def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)

def drive_children(service, parent_id, name=None):
    escaped_parent = parent_id.replace("'", "\\'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace("\\", "\\\\").replace("'", "\\'")
        query.append(f"name = '{escaped_name}'")
    return service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size)",
        pageSize=1000,
    ).execute(num_retries=3).get("files", [])

def ensure_drive_folder(service, parent_id, name, fail_if_exists=False):
    matches = [
        item for item in drive_children(service, parent_id, name)
        if item["mimeType"] == DRIVE_FOLDER_MIME
    ]
    if matches:
        if fail_if_exists:
            raise FileExistsError(f"Drive folder already exists: {name}")
        if len(matches) > 1:
            raise RuntimeError(f"Multiple Drive folders named {name}")
        return matches[0]["id"]
    return service.files().create(
        body={"name": name, "mimeType": DRIVE_FOLDER_MIME, "parents": [parent_id]},
        fields="id",
    ).execute(num_retries=3)["id"]

def drive_upload_file(service, path, parent_id):
    from googleapiclient.http import MediaFileUpload
    path = Path(path)
    media = MediaFileUpload(
        str(path), mimetype="application/octet-stream",
        chunksize=8 * 1024 * 1024, resumable=True,
    )
    request = service.files().create(
        body={"name": path.name, "parents": [parent_id]},
        media_body=media, fields="id,name,size",
    )
    response = None
    while response is None:
        _, response = request.next_chunk(num_retries=5)
    return response

def upload_run_artifacts(paths):
    service = build_drive_service()
    parent = DRIVE_PARENT_FOLDER_ID or get_kaggle_secret("GDRIVE_FOLDER_ID", required=True)
    for folder_name in ("anomaly_detection", MODEL_RUN_NAME):
        parent = ensure_drive_folder(service, parent, folder_name)
    run_folder = ensure_drive_folder(service, parent, RUN_ID, fail_if_exists=True)
    uploaded, failures = [], []
    for path in paths:
        path = Path(path)
        if path.suffix == ".ckpt" and not DRIVE_UPLOAD_CHECKPOINTS:
            continue
        try:
            response = drive_upload_file(service, path, run_folder)
            uploaded.append({
                "name": path.name,
                "drive_file_id": response["id"],
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
        except Exception as error:
            failures.append({"name": path.name, "error": type(error).__name__})
    return service, {
        "status": "complete" if not failures else "partial",
        "drive_folder_id": run_folder,
        "uploaded": uploaded,
        "failures": failures,
    }

def persist_run_artifacts(
    detector, validation_results, test_results, additional_artifacts=()
):
    if not detector.is_fitted:
        raise RuntimeError("Cannot save an unfitted anomaly detector")
    for split_results in (validation_results, test_results):
        if set(split_results) != set(METRIC_NAMES + AUDIT_METRIC_NAMES):
            raise ValueError(f"Unexpected metric keys: {tuple(split_results)}")
    results = {"validation": validation_results, "test": test_results}
    additional_artifacts = [Path(path) for path in additional_artifacts]
    if any(not path.is_file() for path in additional_artifacts):
        raise FileNotFoundError("An additional run artifact is missing")
    run_metadata = {
        "run_id": RUN_ID,
        "model_run_name": MODEL_RUN_NAME,
        "seed": SEED,
        "fit_elapsed_seconds": FIT_ELAPSED_SECONDS,
        "fit_summary": detector.fit_summary,
        "model": {
            "name": MODEL_NAME,
            "pretrained_initialization": PATCHCORE_PRETRAINED,
            **detector.checkpoint_config(),
        },
        "preprocessing": {
            "preset": PATCHCORE_PRESET,
            "pose_crop_enabled": POSE_CROP_ENABLED,
            "pose_crops": POSE_CROPS,
            "input_size": list(IMAGE_SIZE),
            "resize": None if PREPROCESS_RESIZE is None else list(PREPROCESS_RESIZE),
            "resize_shorter_side": PREPROCESS_RESIZE_SHORTER_SIDE,
            "center_crop": None if PREPROCESS_CENTER_CROP is None else list(PREPROCESS_CENTER_CROP),
            "normalize_mean": list(NORMALIZE_MEAN),
            "normalize_std": list(NORMALIZE_STD),
            "train_augmentations_enabled": TRAIN_AUGMENTATIONS_ENABLED,
            "train_augmentation_parameters": {
                "brightness": TRAIN_BRIGHTNESS,
                "contrast": TRAIN_CONTRAST,
                "gamma": TRAIN_GAMMA,
                "saturation": TRAIN_SATURATION,
                "sensor_noise": TRAIN_SENSOR_NOISE,
                "gaussian_noise": TRAIN_GAUSSIAN_NOISE,
                "gaussian_blur": (
                    None if TRAIN_GAUSSIAN_BLUR is None
                    else list(TRAIN_GAUSSIAN_BLUR)
                ),
            },
        },
        "dataset": {
            "manifest_sha256": sha256_file(DATASET_ROOT / "samples.csv"),
            "camera_poses": list(CAMERA_POSES),
            "expected_split_condition_counts": [
                {"split": split, "condition": condition, "count": count}
                for (split, condition), count
                in sorted(EXPECTED_SPLIT_CONDITION_COUNTS.items())
            ],
        },
        "evaluation": {
            "metrics": list(METRIC_NAMES),
            "audit_metrics": list(AUDIT_METRIC_NAMES),
            "image_ranking_uses_raw_scores": True,
            "mask_visibility_audit_enabled": MASK_VISIBILITY_AUDIT_ENABLED,
            "selection": "fixed_reference_configuration",
            "test_used_for_selection": False,
            "pixel_histogram_bins": METRIC_HISTOGRAM_BINS,
            "restrict_pixels_to_target_mask": RESTRICT_PIXELS_TO_TARGET_MASK,
            "diagnostics_enabled": DIAGNOSTICS_ENABLED,
            "diagnostics_group_fields": list(DIAGNOSTICS_GROUP_FIELDS),
            "pro_max_fpr": DIAGNOSTICS_PRO_MAX_FPR,
            "visualization_enabled": VISUALIZATION_ENABLED,
            "visualization_num_clean": VISUALIZATION_NUM_CLEAN,
            "visualization_num_anomalous": VISUALIZATION_NUM_ANOMALOUS,
        },
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    checkpoint_path = detector.save(OUTPUT_DIR / "model.ckpt", metadata=run_metadata)
    metrics_path = save_json_atomic(results, OUTPUT_DIR / "metrics.json")
    # Refresh the final checkpoint size before hashing/uploading its benchmark.
    if BENCHMARK_ENABLED:
        benchmark_results["checkpoint_size_bytes"] = checkpoint_path.stat().st_size
        save_json_atomic(benchmark_results, OUTPUT_DIR / "inference_benchmark_batch1.json")
    manifest_path = save_json_atomic({
        **run_metadata,
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "metrics_sha256": sha256_file(metrics_path),
        "additional_artifacts": [
            {"name": path.name, "size_bytes": path.stat().st_size, "sha256": sha256_file(path)}
            for path in additional_artifacts
        ],
    }, OUTPUT_DIR / "run_manifest.json")
    artifact_paths = [
        checkpoint_path, metrics_path, manifest_path, *additional_artifacts
    ]
    upload_status = {"status": "disabled"}
    if DRIVE_UPLOAD_ENABLED:
        service, upload_status = upload_run_artifacts(artifact_paths)
        status_path = save_json_atomic(upload_status, OUTPUT_DIR / "upload_status.json")
        drive_upload_file(service, status_path, upload_status["drive_folder_id"])
    else:
        save_json_atomic(upload_status, OUTPUT_DIR / "upload_status.json")
    return {"artifacts": artifact_paths, "upload": upload_status}


In [ ]:
from __future__ import annotations

import threading

def _latency_summary(milliseconds: list[float]) -> dict[str, float]:
    if not milliseconds:
        raise ValueError("Latency samples cannot be empty")
    ordered = sorted(float(value) for value in milliseconds)

    def percentile(fraction: float) -> float:
        position = fraction * (len(ordered) - 1)
        lower, upper = math.floor(position), math.ceil(position)
        weight = position - lower
        return ordered[lower] * (1 - weight) + ordered[upper] * weight

    return {
        "minimum_ms": ordered[0],
        "mean_ms": sum(ordered) / len(ordered),
        "median_ms": percentile(0.50),
        "p90_ms": percentile(0.90),
        "p95_ms": percentile(0.95),
        "maximum_ms": ordered[-1],
    }


def _cuda_memory_baseline(device: torch.device) -> tuple[int, int] | None:
    if device.type != "cuda":
        return None
    torch.cuda.synchronize(device)
    torch.cuda.empty_cache()
    baseline = (
        torch.cuda.memory_allocated(device), torch.cuda.memory_reserved(device)
    )
    torch.cuda.reset_peak_memory_stats(device)
    return baseline


def _cuda_memory_report(
    device: torch.device, baseline: tuple[int, int] | None,
) -> dict[str, float] | None:
    if baseline is None:
        return None
    allocated, reserved = baseline
    peak_allocated = torch.cuda.max_memory_allocated(device)
    peak_reserved = torch.cuda.max_memory_reserved(device)
    return {
        "baseline_allocated_mib": allocated / 2**20,
        "baseline_reserved_mib": reserved / 2**20,
        "peak_allocated_mib": peak_allocated / 2**20,
        "peak_reserved_mib": peak_reserved / 2**20,
        "incremental_peak_allocated_mib": (peak_allocated - allocated) / 2**20,
        "incremental_peak_reserved_mib": (peak_reserved - reserved) / 2**20,
    }


@torch.inference_mode()
def benchmark_anomaly_inference(
    detector: Any,
    loader: Any,
    *,
    device: torch.device | str,
    warmup_runs: int = 10,
    timed_runs: int = 50,
    split: str = "validation",
) -> dict[str, Any]:
    """Measure predict-only and sequential batch-one pipeline latency.

    ``predict_only`` reuses one preprocessed, device-resident input. The other
    measurements include the loader's complete work, which for the synthetic
    dataset includes annotation I/O and aligned mask transforms. They are not
    a mask-free deployment benchmark. CUDA allocator peaks include all live
    tensors on the selected device, not just the model, and exclude driver
    allocations. No power sensor or energy estimate is used.
    """
    if getattr(loader, "batch_size", None) != 1:
        raise ValueError("Inference benchmark loader must use batch_size=1")
    if getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Sequential inference benchmark requires num_workers=0")
    if not isinstance(warmup_runs, int) or not isinstance(timed_runs, int):
        raise TypeError("warmup_runs and timed_runs must be integers")
    if warmup_runs < 1 or timed_runs < 1:
        raise ValueError("warmup_runs and timed_runs must be positive")
    if not detector.is_fitted:
        raise RuntimeError("Fit or restore the detector before benchmarking")

    device = torch.device(device)
    uses_cuda = device.type == "cuda"
    was_training = detector.training
    detector.to(device)
    detector.eval()

    def synchronize() -> None:
        if uses_cuda:
            torch.cuda.synchronize(device)

    def next_batch(iterator):
        try:
            return next(iterator), iterator
        except StopIteration:
            iterator = iter(loader)
            try:
                return next(iterator), iterator
            except StopIteration as error:
                raise ValueError("Inference benchmark loader is empty") from error

    try:
        iterator = iter(loader)
        batch, iterator = next_batch(iterator)
        images = batch["image"].to(device, non_blocking=uses_cuda)
        if images.ndim != 4 or images.shape[0] != 1:
            raise ValueError("Inference benchmark requires one BCHW image")
        input_shape, input_dtype = list(images.shape), str(images.dtype)
        for _ in range(warmup_runs):
            prediction = detector.predict(images)
            del prediction
        synchronize()
        baseline = _cuda_memory_baseline(device)
        predict_ms = []
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            prediction = detector.predict(images)
            synchronize()
            predict_ms.append((time.perf_counter() - start) * 1_000)
            # Do not retain the previous output across the next forward.
            del prediction
        predict_memory = _cuda_memory_report(device, baseline)
        del images, batch, iterator

        iterator = iter(loader)
        for _ in range(warmup_runs):
            batch, iterator = next_batch(iterator)
            images = batch["image"].to(device, non_blocking=uses_cuda)
            prediction = detector.predict(images)
            synchronize()
            del prediction, images, batch
        baseline = _cuda_memory_baseline(device)
        preprocessing_ms, accelerator_ms, pipeline_ms = [], [], []
        iterator = iter(loader)
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            batch, iterator = next_batch(iterator)
            preprocessing_end = time.perf_counter()
            images = batch["image"].to(device, non_blocking=uses_cuda)
            if list(images.shape) != input_shape or str(images.dtype) != input_dtype:
                raise ValueError("Benchmark inputs must have one stable shape and dtype")
            prediction = detector.predict(images)
            synchronize()
            end = time.perf_counter()
            preprocessing_ms.append((preprocessing_end - start) * 1_000)
            accelerator_ms.append((end - preprocessing_end) * 1_000)
            pipeline_ms.append((end - start) * 1_000)
            del prediction, images, batch
        pipeline_memory = _cuda_memory_report(device, baseline)

        predict_summary = _latency_summary(predict_ms)
        accelerator_summary = _latency_summary(accelerator_ms)
        pipeline_summary = _latency_summary(pipeline_ms)
        parameters = list(detector.parameters())
        buffers = list(detector.buffers())
        bank = getattr(detector, "memory_bank", None)
        return {
            "schema_version": 1,
            "scope": (
                "predict_only: one device-resident preprocessed image; pipeline: "
                "sequential loader (including annotation I/O and mask transforms), "
                "host-to-device transfer and detector.predict; no worker prefetch"
            ),
            "memory_scope": (
                "CUDA allocator totals include the model, memory bank and other live "
                "tensors on this device; driver allocations are excluded"
            ),
            "split": split,
            "batch_size": 1,
            "warmup_runs": warmup_runs,
            "timed_runs": timed_runs,
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(device) if uses_cuda else None,
            "input_shape": input_shape,
            "input_dtype": input_dtype,
            "model": type(detector).__name__,
            "gaussian_sigma": getattr(detector, "gaussian_sigma", None),
            "total_parameters": sum(parameter.numel() for parameter in parameters),
            "trainable_parameters": sum(
                parameter.numel() for parameter in parameters if parameter.requires_grad
            ),
            "parameters_mib": sum(
                parameter.numel() * parameter.element_size() for parameter in parameters
            ) / 2**20,
            "buffers_mib": sum(
                buffer.numel() * buffer.element_size() for buffer in buffers
            ) / 2**20,
            "memory_bank_shape": list(bank.shape) if bank is not None else None,
            "memory_bank_mib": (
                bank.numel() * bank.element_size() / 2**20 if bank is not None else None
            ),
            "latency": {
                "predict_only": predict_summary,
                "decode_and_preprocess": _latency_summary(preprocessing_ms),
                "host_to_device_and_predict": accelerator_summary,
                "end_to_end": pipeline_summary,
            },
            "throughput_images_per_second": 1_000 / accelerator_summary["mean_ms"],
            "predict_only_images_per_second": 1_000 / predict_summary["mean_ms"],
            "end_to_end_images_per_second": 1_000 / pipeline_summary["mean_ms"],
            "cuda_memory": pipeline_memory,
            "predict_only_cuda_memory": predict_memory,
            "environment": {
                "python_version": sys.version.split()[0],
                "platform": platform.platform(),
                "torch_version": str(torch.__version__),
                "cuda_version": torch.version.cuda,
                "cudnn_version": torch.backends.cudnn.version(),
                "cudnn_benchmark": torch.backends.cudnn.benchmark,
                "cudnn_deterministic": torch.backends.cudnn.deterministic,
                "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
                "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
                "float32_matmul_precision": torch.get_float32_matmul_precision(),
            },
            "energy_joules_per_image": None,
            "energy_note": "Not measured; latency alone is not an energy measurement",
        }
    finally:
        detector.train(was_training)

def _nvml_handle_for_cuda(nvml: Any, device: torch.device) -> tuple[Any, str]:
    """Match by CUDA UUID; never assume CUDA and NVML ordinals are identical."""
    properties = torch.cuda.get_device_properties(device)
    cuda_uuid = getattr(properties, "uuid", None)
    if cuda_uuid:
        uuid_text = cuda_uuid.decode("ascii") if isinstance(cuda_uuid, bytes) else str(cuda_uuid)
        if not uuid_text.startswith(("GPU-", "MIG-")):
            uuid_text = "GPU-" + uuid_text
        try:
            return nvml.nvmlDeviceGetHandleByUUID(uuid_text), "cuda_uuid"
        except Exception as error:
            raise RuntimeError(f"Cannot match the CUDA UUID to NVML: {error}") from error
    if nvml.nvmlDeviceGetCount() == 1:
        return nvml.nvmlDeviceGetHandleByIndex(0), "single_nvml_device"
    raise RuntimeError(
        "CUDA UUID is unavailable and multiple NVML GPUs exist; "
        "refusing to measure an ambiguous physical device"
    )


def _nvml_counter_energy_window(
    predict_once: Any, nvml: Any, handle: Any, duration_seconds: float,
) -> dict[str, Any]:
    """Counter units are mJ; inference failures propagate rather than being hidden."""
    method = "nvml_total_energy_counter"
    try:
        before = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
        if before < 0:
            raise ValueError("negative energy counter")
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    count = 0
    while count == 0 or time.perf_counter() - start < duration_seconds:
        predict_once()
        count += 1
    try:
        after = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    elapsed = time.perf_counter() - start
    if after <= before:
        return {
            "status": "unavailable", "method": method,
            "reason": "Energy counter did not advance or was reset during measurement",
        }
    joules = (after - before) / 1_000.0
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "counter_start_millijoules": before,
        "counter_end_millijoules": after,
        "power_samples": [],
    }


def _nvml_power_energy_window(
    predict_once: Any, nvml: Any, handle: Any,
    duration_seconds: float, sample_interval_seconds: float,
) -> dict[str, Any]:
    """Integrate measured GPU power, with explicit boundary samples and no TDP estimate."""
    method = "nvml_power_integration"

    def read_watts() -> float:
        watts = float(nvml.nvmlDeviceGetPowerUsage(handle)) / 1_000.0
        if not math.isfinite(watts) or watts <= 0:
            raise ValueError("NVML power reading must be positive and finite")
        return watts

    try:
        first_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    samples = [(start, first_watts)]
    errors = []
    stop = threading.Event()

    def sample_power() -> None:
        while not stop.wait(sample_interval_seconds):
            try:
                watts = read_watts()
                samples.append((time.perf_counter(), watts))
            except Exception as error:
                errors.append(str(error))
                return

    worker = threading.Thread(target=sample_power, daemon=True)
    worker.start()
    count = 0
    try:
        while count == 0 or time.perf_counter() - start < duration_seconds:
            predict_once()
            count += 1
            if errors:
                break
    finally:
        stop.set()
        worker.join()
    if errors:
        return {"status": "unavailable", "method": method, "reason": errors[0]}
    try:
        final_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    samples.append((time.perf_counter(), final_watts))
    elapsed = samples[-1][0] - start
    # Two endpoints alone are not adequate for the sampled-power fallback.
    if len(samples) < 3:
        return {
            "status": "unavailable", "method": method,
            "reason": "Too few power samples; increase the measurement window",
        }
    joules = sum(
        (right[0] - left[0]) * (left[1] + right[1]) * 0.5
        for left, right in zip(samples, samples[1:])
    )
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "power_sample_interval_seconds_requested": sample_interval_seconds,
        "power_sample_count": len(samples),
        "power_samples": [
            {"elapsed_seconds": timestamp - start, "power_watts": watts}
            for timestamp, watts in samples
        ],
    }


@torch.inference_mode()
def measure_anomaly_inference_energy(
    detector: Any, loader: Any, *, device: torch.device | str,
    repeats: int = 3, duration_seconds: float = 30.0,
    warmup_seconds: float = 5.0, sample_interval_seconds: float = 0.1,
    method: str = "auto",
) -> dict[str, Any]:
    """Measure device-wide GPU J/image in a separate sustained predict-only phase.

    The model and one preprocessed input are already GPU resident. Each complete
    predict is synchronized, and its output discarded. Includes GPU idle/boundary
    energy, excludes CPU, I/O and host-to-device energy. Raw total energy is not
    idle-subtracted and is not attributed to an individual process. Sensor errors
    yield an explicit unavailable/partial report; prediction errors propagate.
    """
    if not isinstance(repeats, int) or isinstance(repeats, bool) or repeats < 1:
        raise ValueError("Energy repeats must be a positive integer")
    for name, value in (
        ("duration_seconds", duration_seconds),
        ("sample_interval_seconds", sample_interval_seconds),
    ):
        if not math.isfinite(value) or value <= 0:
            raise ValueError(f"{name} must be positive and finite")
    if not math.isfinite(warmup_seconds) or warmup_seconds < 0:
        raise ValueError("warmup_seconds must be non-negative and finite")
    if method not in {"auto", "counter", "power"}:
        raise ValueError("Energy method must be auto, counter or power")
    if getattr(loader, "batch_size", None) != 1 or getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Energy requires batch_size=1 and num_workers=0")
    if not detector.is_fitted:
        raise RuntimeError("Restore the detector before measuring energy")
    device = torch.device(device)
    report = {
        "schema_version": 1,
        "status": "unavailable", "reason": None, "method": None,
        "metric": "device_wide_gpu_energy_per_inference",
        "scope": (
            "predict-only repeated synchronized inference; model and one preprocessed "
            "image are CUDA resident; excludes input transfer, CPU and loader work"
        ),
        "device": str(device), "batch_size": 1,
        "repeats_requested": repeats,
        "duration_seconds_per_repeat_requested": duration_seconds,
        "warmup_seconds_requested": warmup_seconds,
        "sample_interval_seconds_requested": sample_interval_seconds,
        "method_requested": method,
        "idle_subtracted": False, "process_attributed": False,
        "energy_joules_per_image": None,
        "average_gpu_power_watts": None,
        "energy_joules_per_image_std": None,
        "total_inferences": 0, "runs": [], "warnings": [],
    }
    if device.type != "cuda" or not torch.cuda.is_available():
        report["reason"] = "CUDA GPU is required for NVML energy measurements"
        return report
    try:
        import pynvml
    except ImportError:
        report["reason"] = "Install nvidia-ml-py (import pynvml) to read GPU energy/power"
        return report
    try:
        pynvml.nvmlInit()
    except Exception as error:
        report["reason"] = f"NVML initialization failed: {error}"
        return report

    was_training = detector.training
    try:
        try:
            handle, mapping = _nvml_handle_for_cuda(pynvml, device)
            report["device_mapping"] = mapping
            uuid = pynvml.nvmlDeviceGetUUID(handle)
            report["nvml_gpu_uuid"] = uuid.decode() if isinstance(uuid, bytes) else str(uuid)
            name = pynvml.nvmlDeviceGetName(handle)
            report["gpu_name"] = name.decode() if isinstance(name, bytes) else str(name)
        except Exception as error:
            report["reason"] = f"NVML device mapping failed: {error}"
            return report
        try:
            processes = pynvml.nvmlDeviceGetComputeRunningProcesses(handle)
            report["gpu_compute_process_count_at_start"] = len(processes)
            if len(processes) > 1:
                report["warnings"].append(
                    "Multiple compute processes are visible on this GPU; "
                    "device-wide energy cannot be assigned exclusively to this model"
                )
        except Exception:
            report["gpu_compute_process_count_at_start"] = None
        try:
            report["power_limit_watts"] = pynvml.nvmlDeviceGetPowerManagementLimit(handle) / 1_000.0
        except Exception:
            report["power_limit_watts"] = None
        detector.to(device).eval()
        try:
            batch = next(iter(loader))
        except StopIteration as error:
            raise ValueError("Energy measurement loader is empty") from error
        images = batch["image"].to(device)
        del batch
        if images.ndim != 4 or images.shape[0] != 1 or images.dtype != torch.float32:
            raise ValueError("Energy comparison requires one FP32 BCHW image")
        report["input_shape"] = list(images.shape)
        report["input_dtype"] = str(images.dtype)

        def predict_once() -> None:
            prediction = detector.predict(images)
            torch.cuda.synchronize(device)
            del prediction

        torch.cuda.synchronize(device)
        warmup_start = time.perf_counter()
        predict_once()
        while time.perf_counter() - warmup_start < warmup_seconds:
            predict_once()
        report["warmup_seconds_actual"] = time.perf_counter() - warmup_start
        chosen_method = "power" if method == "power" else "counter"
        for repeat in range(repeats):
            if chosen_method == "counter":
                run = _nvml_counter_energy_window(
                    predict_once, pynvml, handle, duration_seconds
                )
                # Auto chooses one successful backend on the first repeat only;
                # never average counter and integrated-power repeats together.
                if repeat == 0 and method == "auto" and run["status"] != "measured":
                    report["warnings"].append("Counter unavailable: " + str(run.get("reason")))
                    chosen_method = "power"
                    run = _nvml_power_energy_window(
                        predict_once, pynvml, handle,
                        duration_seconds, sample_interval_seconds,
                    )
            else:
                run = _nvml_power_energy_window(
                    predict_once, pynvml, handle,
                    duration_seconds, sample_interval_seconds,
                )
            run["repeat"] = repeat + 1
            report["runs"].append(run)
            if run["status"] != "measured":
                report["status"] = "partial" if repeat else "unavailable"
                report["reason"] = run.get("reason", "GPU energy sensor failed")
                return report
        runs = report["runs"]
        total_images = sum(run["inferences"] for run in runs)
        total_joules = sum(run["gpu_energy_joules"] for run in runs)
        total_seconds = sum(run["duration_seconds"] for run in runs)
        values = [run["energy_joules_per_image"] for run in runs]
        mean = sum(values) / len(values)
        report.update({
            "status": "measured", "reason": None,
            "method": runs[0]["method"],
            "total_inferences": total_images,
            "total_gpu_energy_joules": total_joules,
            "total_duration_seconds": total_seconds,
            "energy_joules_per_image": total_joules / total_images,
            "average_gpu_power_watts": total_joules / total_seconds,
            "energy_joules_per_image_std": (
                math.sqrt(sum((value - mean) ** 2 for value in values) / (len(values) - 1))
                if len(values) > 1 else 0.0
            ),
        })
        return report
    finally:
        detector.train(was_training)
        pynvml.nvmlShutdown()


## Experiment protocol

Pose filtering is applied to all three splits. Expected counts: train 2,800 clean;
validation 296 clean + 96 holes; test 404 clean + 404 holes. Clean–hole pairs stay
inside their original splits; training contains no anomalous images. The full
manifest is validated before selecting the pose-A subset.

Pre-fit QA follows the existing notebooks: a non-blocking visual preprocessing audit.
`MASK_VISIBILITY_AUDIT_ENABLED=False` skips the additional whole-subset mask scan.
When enabled, this diagnostic saves pixel counts before/after the pose crop and Reference
geometry and prints a warning for empty masks; it never stops training or filters rows.

The fixed Reference center crop may hide edge defects. In that case, image-level labels
remain anomalous while transformed defect masks can be empty. This is a preprocessing
limitation, not a data-loading exception; it can affect measured detection performance.
Neither the geometry nor sample selection is changed using validation/test annotations.

The efficiency report has two distinct scopes:

- **Predict only:** preprocessed float32 input already resident on the GPU;
  includes backbone, nearest-neighbour search, map interpolation/smoothing and output validation.
- **Sequential pipeline:** dataset decode/preprocessing, host-to-device transfer and predict.
  This dataset loader also reads annotation masks, so this is **not** a mask-free deployment timing.

Both use batch 1, 10 warm-up and 50 timed iterations, with CUDA synchronization and
no gradients. Peak allocated/reserved GPU memory includes the memory bank and
other live CUDA tensors; incremental peak is listed separately. Checkpoint size,
bank size, parameter count, versions and precision settings are saved. Energy is
not measured. Keep hardware, chunk sizes and timing scope fixed when comparing models.
The predict-only stage reuses one validation input; the pipeline stage traverses
validation in manifest order. It does not assess the accuracy of a subset.

### GPU energy (separate from the latency windows)

The default is three **30 s** sustained predict-only windows after a **5 s**
warm-up, batch 1, FP32, and the same preprocessed image already on the GPU.
The complete predict call and its postprocessing run every iteration; outputs
are discarded. No feature cache, refit, recalibration or input shortcut is used.

NVML's total-energy counter is preferred (mJ converted to J). When unavailable
or non-advancing, the first window chooses the measured-power fallback:
100 ms requested sampling and trapezoidal integration. A single backend is
used for all successful repeats; a partial sensor failure leaves the aggregate
null and records the reason. Inference errors still propagate.

The result is **device-wide GPU J/image**, including GPU idle power and memory,
not per-process or whole-system energy. CPU, loader and input-transfer work are
excluded; idle power is not subtracted. The CUDA/NVML devices are matched by
UUID, and additional visible compute processes are flagged. Use a fresh T4
session with no other model resident for the comparison. Actual power and
sensor readings are saved, never estimated from nominal GPU power or latency.

The energy phase adds about 95 seconds when the counter succeeds. Read the
**final saved JSON or the post-energy printout**, not the earlier latency
printout whose energy field may still be null.

### Reuse an already-fitted live session

Run the updated **batch-one-efficiency-definitions**, **gpu-energy-dependencies**
and **gpu-energy-run** cells. They reuse the existing model, preprocessing and
run directory; the energy cell supplies its defaults if the older Configuration
cell did not define them. Do not rerun Configuration, run initialization or the
fit cell solely to add energy: that would change the run identity or refit the
bank. The checkpoint, bank and calibration are not changed by energy measurement.
A fresh session running all cells still performs the original clean-only fit.


## End-to-end execution

Run the remaining cells in order: initialize, prepare and verify data, audit preprocessing, build and fit the model, evaluate, inspect diagnostics, then persist artifacts.

In [ ]:
from __future__ import annotations

# Start execution only after every class and function has been defined.
if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU before fitting Reference with a 50k bank")
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
DATASET_ROOT = (
    Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
    if DATASET_ROOT_OVERRIDE is not None else find_dataset_root()
)
preprocessing_paths = []
benchmark_paths = []
print(f"Dataset: {DATASET_ROOT}")
print(f"Camera poses: {CAMERA_POSES}")
print(f"Device: {DEVICE}")
print(f"Artifact output directory: {OUTPUT_DIR}")


In [ ]:
from __future__ import annotations

data = prepare_data(DATASET_ROOT)
TRAIN_PREPROCESSING_CONFIG = data["train_config"]
TRAIN_PREPROCESSING = data["train_preprocessing"]
EVALUATION_PREPROCESSING_CONFIG = data["evaluation_config"]
EVALUATION_PREPROCESSING = data["evaluation_preprocessing"]
train_loader = data["train_loader"]
validation_loader = data["validation_loader"]
test_loader = data["test_loader"]
loaders = data["loaders"]
for split, loader in loaders:
    print(f"{split:10s}: {len(loader.dataset)} samples, {len(loader)} batches")
print("DataLoaders ready")

In [ ]:
from __future__ import annotations

smoke_test_loaders(loaders)
print("Loading verified")

In [ ]:
from __future__ import annotations

# Existing notebooks use visual QA only; extra mask diagnostics are opt-in.
if MASK_VISIBILITY_AUDIT_ENABLED:
    visibility_report = audit_processed_hole_visibility(loaders)
    visibility_path = save_json_atomic(
        visibility_report, OUTPUT_DIR / "preprocessing_validity_audit.json"
    )
    preprocessing_paths.append(visibility_path)
    print({
        "audited_holes": visibility_report["hole_count"],
        "minimum_processed_hole_pixels": visibility_report["minimum_processed_hole_pixels"],
        "partially_clipped_holes": len(visibility_report["partially_clipped_holes"]),
        "empty_processed_holes": visibility_report["empty_processed_holes"],
    })
    if visibility_report["empty_processed_holes"]:
        print(
            "WARNING: preprocessing leaves an empty defect or target-wheel mask "
            f"in {len(visibility_report['empty_processed_holes'])} samples. "
            "Diagnostics saved; continuing with unchanged rows, labels and geometry."
        )
else:
    print("Numerical mask visibility diagnostics disabled; no samples filtered.")
if AUDIT_ENABLED:
    audit_dataset = dict(loaders)[AUDIT_SPLIT].dataset
    audit_indices = AUDIT_INDICES
    if audit_indices is None:
        audit_indices = tuple(
            next(index for index, row in enumerate(audit_dataset.rows)
                 if row["condition"] == condition)
            for condition in ("clean", "hole")
        )
    audit_figure = audit_preprocessing(
        DATASET_ROOT, EVALUATION_PREPROCESSING_CONFIG,
        split=AUDIT_SPLIT, indices=audit_indices,
        camera_poses=CAMERA_POSES, variants=AUDIT_VARIANTS, seed=SEED,
    )
    audit_path = OUTPUT_DIR / "preprocessing_examples.png"
    audit_figure.savefig(audit_path, dpi=VISUALIZATION_DPI)
    preprocessing_paths.append(audit_path)
    plt.show()
    plt.close(audit_figure)


In [ ]:
from __future__ import annotations

model = build_model()
optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(
    f"Model: {MODEL_NAME} | backbone={PATCHCORE_BACKBONE} | "
    f"parameters={total_parameters:,} | trainable={trainable_parameters:,} | "
    f"optimizer={OPTIMIZER_NAME} | scheduler={SCHEDULER_NAME}"
)

In [ ]:
from __future__ import annotations

print(f"Fitting {MODEL_RUN_NAME} on {DEVICE}: clean training images only")
fit_start = time.perf_counter()
model.fit(train_loader, device=DEVICE)
if DEVICE.type == "cuda":
    torch.cuda.synchronize(DEVICE)
FIT_ELAPSED_SECONDS = time.perf_counter() - fit_start
if model.fit_summary["num_training_images"] != len(train_loader.dataset):
    raise RuntimeError("Fit did not use the complete pose-A clean training split")
if model.fit_summary["memory_bank_embeddings"] != PATCHCORE_MAX_MEMORY_BANK_SIZE:
    raise RuntimeError("Actual memory-bank size differs from the requested 50k configuration")
fit_report = {
    "run_id": RUN_ID,
    "camera_poses": list(CAMERA_POSES),
    "pose_crops": POSE_CROPS,
    "input_size": list(IMAGE_SIZE),
    "manifest_sha256": sha256_file(DATASET_ROOT / "samples.csv"),
    "fit_elapsed_seconds": FIT_ELAPSED_SECONDS,
    "model_config": model.checkpoint_config(),
    "fit_summary": model.fit_summary,
}
fit_path = save_json_atomic(fit_report, OUTPUT_DIR / "fit_summary.json")
preprocessing_paths.append(fit_path)
model.save(OUTPUT_DIR / "model.ckpt", metadata=fit_report)
print(json.dumps(fit_report, indent=2))
print("Post-fit checkpoint saved; metric evaluation does not update the memory bank")


In [ ]:
from __future__ import annotations

benchmark_results = {}
if BENCHMARK_ENABLED:
    print("Benchmarking fitted PatchCore: batch 1, no parameter selection")
    benchmark_loader = build_dataloader(
        DATASET_ROOT, BENCHMARK_SPLIT,
        batch_size=1, num_workers=0, pin_memory=PIN_MEMORY, seed=SEED,
        preprocessing=EVALUATION_PREPROCESSING, camera_poses=CAMERA_POSES,
    )
    benchmark_results = benchmark_anomaly_inference(
        model, benchmark_loader, device=DEVICE,
        warmup_runs=BENCHMARK_WARMUP_RUNS, timed_runs=BENCHMARK_TIMED_RUNS,
        split=BENCHMARK_SPLIT,
    )
    benchmark_results.update({
        "run_id": RUN_ID,
        "camera_poses": list(CAMERA_POSES),
        "pose_crops": POSE_CROPS,
        "manifest_sha256": sha256_file(DATASET_ROOT / "samples.csv"),
        "fit_elapsed_seconds": FIT_ELAPSED_SECONDS,
        "fit_summary": model.fit_summary,
        "checkpoint_config": model.checkpoint_config(),
        "checkpoint_size_bytes": (OUTPUT_DIR / "model.ckpt").stat().st_size,
    })
    benchmark_path = save_json_atomic(
        benchmark_results, OUTPUT_DIR / "inference_benchmark_batch1.json"
    )
    benchmark_paths.append(benchmark_path)
    print(json.dumps(benchmark_results, indent=2))
else:
    print("Efficiency benchmark disabled")


In [ ]:
from __future__ import annotations

# Defaults also allow this cell to run in an already-fitted live session,
# without rerunning Configuration and changing the original run/output identity.
ENERGY_ENABLED = globals().get("ENERGY_ENABLED", True)
ENERGY_METHOD = globals().get("ENERGY_METHOD", "auto")
ENERGY_REPEATS = globals().get("ENERGY_REPEATS", 3)
ENERGY_DURATION_SECONDS = globals().get("ENERGY_DURATION_SECONDS", 30.0)
ENERGY_WARMUP_SECONDS = globals().get("ENERGY_WARMUP_SECONDS", 5.0)
ENERGY_POWER_SAMPLE_INTERVAL_SECONDS = globals().get("ENERGY_POWER_SAMPLE_INTERVAL_SECONDS", 0.1)
benchmark_results = globals().get("benchmark_results", {})
benchmark_paths = globals().get("benchmark_paths", [])

if ENERGY_ENABLED:
    print(
        f"Measuring PatchCore GPU energy: {ENERGY_REPEATS} x {ENERGY_DURATION_SECONDS:g} s "
        f"after {ENERGY_WARMUP_SECONDS:g} s warm-up; predict-only, batch 1, FP32"
    )
    # Independent of BENCHMARK_ENABLED: energy-only reruns need no latency rerun.
    energy_loader = build_dataloader(
        DATASET_ROOT, BENCHMARK_SPLIT,
        batch_size=1, num_workers=0, pin_memory=False, seed=SEED,
        preprocessing=EVALUATION_PREPROCESSING, camera_poses=CAMERA_POSES,
    )
    energy_report = measure_anomaly_inference_energy(
        model, energy_loader, device=DEVICE,
        repeats=ENERGY_REPEATS,
        duration_seconds=ENERGY_DURATION_SECONDS,
        warmup_seconds=ENERGY_WARMUP_SECONDS,
        sample_interval_seconds=ENERGY_POWER_SAMPLE_INTERVAL_SECONDS,
        method=ENERGY_METHOD,
    )
else:
    energy_report = {
        "schema_version": 1, "status": "disabled",
        "reason": "ENERGY_ENABLED=False", "method": None,
        "energy_joules_per_image": None,
        "average_gpu_power_watts": None,
        "energy_joules_per_image_std": None,
        "runs": [], "warnings": [],
    }
energy_report.update({
    "run_id": RUN_ID,
    "model": type(model).__name__,
    "manifest_sha256": sha256_file(DATASET_ROOT / "samples.csv"),
    "camera_poses": list(CAMERA_POSES),
    "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
    "checkpoint_config": model.checkpoint_config(),
    "fit_summary": model.fit_summary,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_performed_in_energy_stage": False,
    "calibration_performed_in_energy_stage": False,
})
energy_path = save_json_atomic(energy_report, OUTPUT_DIR / "inference_energy_batch1.json")
if energy_path not in benchmark_paths:
    benchmark_paths.append(energy_path)
if benchmark_results:
    benchmark_results["gpu_energy_measurement"] = energy_report
    benchmark_results["energy_joules_per_image"] = energy_report["energy_joules_per_image"]
    benchmark_results["energy_note"] = (
        "Measured device-wide GPU energy per predict-only inference via "
        + str(energy_report["method"])
        + "; excludes CPU/input transfer/loader; idle is not subtracted"
        if energy_report["status"] == "measured"
        else "GPU energy not measured completely: " + str(energy_report.get("reason"))
    )
    benchmark_path = save_json_atomic(
        benchmark_results, OUTPUT_DIR / "inference_benchmark_batch1.json"
    )
    if benchmark_path not in benchmark_paths:
        benchmark_paths.append(benchmark_path)
print({
    "energy_status": energy_report["status"],
    "energy_method": energy_report["method"],
    "gpu_energy_joules_per_image": energy_report["energy_joules_per_image"],
    "average_gpu_power_watts": energy_report["average_gpu_power_watts"],
    "energy_joules_per_image_std": energy_report["energy_joules_per_image_std"],
    "warnings": energy_report["warnings"],
    "reason": energy_report.get("reason"),
})
print(f"Energy report: {energy_path}")
if benchmark_results:
    print("Updated benchmark JSON (includes the energy phase):")
    print(json.dumps(benchmark_results, indent=2, allow_nan=False))
if energy_report["status"] not in {"measured", "disabled"}:
    print("WARNING: energy could not be measured reliably; inspect inference_energy_batch1.json")


In [ ]:
from __future__ import annotations

validation_diagnostics = (
    AnomalyDiagnostics(
        validation_loader.dataset.rows,
        preprocessor=EVALUATION_PREPROCESSING,
        histogram_bins=METRIC_HISTOGRAM_BINS,
        pro_bins=DIAGNOSTICS_PRO_BINS,
        pro_max_fpr=DIAGNOSTICS_PRO_MAX_FPR,
        group_fields=DIAGNOSTICS_GROUP_FIELDS,
        num_extreme_examples=DIAGNOSTICS_NUM_EXTREME_EXAMPLES,
        restrict_pixels_to_target_mask=RESTRICT_PIXELS_TO_TARGET_MASK,
    )
    if DIAGNOSTICS_ENABLED else None
)
print("[2/6] Evaluating validation split")
validation_metrics = evaluate_anomaly_detector(
    model, validation_loader, device=DEVICE,
    histogram_bins=METRIC_HISTOGRAM_BINS,
    restrict_pixels_to_target_mask=RESTRICT_PIXELS_TO_TARGET_MASK,
    description="Validation", diagnostics=validation_diagnostics
)
print(validation_metrics)


In [ ]:
from __future__ import annotations

test_diagnostics = (
    AnomalyDiagnostics(
        test_loader.dataset.rows,
        preprocessor=EVALUATION_PREPROCESSING,
        histogram_bins=METRIC_HISTOGRAM_BINS,
        pro_bins=DIAGNOSTICS_PRO_BINS,
        pro_max_fpr=DIAGNOSTICS_PRO_MAX_FPR,
        group_fields=DIAGNOSTICS_GROUP_FIELDS,
        num_extreme_examples=DIAGNOSTICS_NUM_EXTREME_EXAMPLES,
        restrict_pixels_to_target_mask=RESTRICT_PIXELS_TO_TARGET_MASK,
    )
    if DIAGNOSTICS_ENABLED else None
)
print("[3/6] Evaluating test split")
test_metrics = evaluate_anomaly_detector(
    model, test_loader, device=DEVICE,
    histogram_bins=METRIC_HISTOGRAM_BINS,
    restrict_pixels_to_target_mask=RESTRICT_PIXELS_TO_TARGET_MASK,
    description="Test", diagnostics=test_diagnostics
)
print(test_metrics)


In [ ]:
from __future__ import annotations

diagnostics_results = {}
diagnostics_paths = []
if DIAGNOSTICS_ENABLED:
    print("[4/6] Saving score, failure-case, grouped AP, and PRO diagnostics")
    for split, split_diagnostics in (
        ("validation", validation_diagnostics),
        ("test", test_diagnostics),
    ):
        diagnostic_result, split_paths = save_anomaly_diagnostics(
            split_diagnostics,
            OUTPUT_DIR,
            split=split,
            colormap=VISUALIZATION_COLORMAP,
            overlay_alpha=VISUALIZATION_OVERLAY_ALPHA,
            dpi=VISUALIZATION_DPI,
        )
        diagnostics_results[split] = diagnostic_result
        diagnostics_paths.extend(split_paths)
        for path in split_paths:
            if path.suffix.lower() == ".png":
                with Image.open(path) as preview:
                    display(preview.copy())
    print(json.dumps(diagnostics_results, indent=2))
else:
    print("Extended diagnostics disabled")


In [ ]:
from __future__ import annotations

visualization_paths = []
if VISUALIZATION_ENABLED:
    print("[5/6] Creating qualitative anomaly visualizations")
    for split, loader in (("validation", validation_loader), ("test", test_loader)):
        visualization_path = save_anomaly_visualizations(
            model,
            loader,
            OUTPUT_DIR / f"{split}_examples.png",
            device=DEVICE,
            preprocessor=EVALUATION_PREPROCESSING,
            title=f"{split.title()} anomaly examples",
            num_clean=VISUALIZATION_NUM_CLEAN,
            num_anomalous=VISUALIZATION_NUM_ANOMALOUS,
            colormap=VISUALIZATION_COLORMAP,
            overlay_alpha=VISUALIZATION_OVERLAY_ALPHA,
            dpi=VISUALIZATION_DPI,
        )
        visualization_paths.append(visualization_path)
        with Image.open(visualization_path) as preview:
            display(preview.copy())
else:
    print("Qualitative visualization disabled")

In [ ]:
from __future__ import annotations

print("Saving complete run artifacts")
saved = persist_run_artifacts(
    model, validation_metrics, test_metrics,
    additional_artifacts=[
        *preprocessing_paths, *benchmark_paths, *visualization_paths, *diagnostics_paths,
    ],
)
print(json.dumps(saved, indent=2, default=str))


## Outputs

The unique folder under `/kaggle/working/` contains:

- `model.ckpt`, `fit_summary.json`, `run_manifest.json` and `metrics.json`;
- `inference_benchmark_batch1.json`: latencies, throughput, GPU allocator peaks,
  static parameter/buffer/bank sizes, checkpoint size, hardware/software metadata and the nested GPU-energy report;
- `inference_energy_batch1.json`: device-wide GPU J/image, mean power, repeat variability, sensor readings, GPU identity and explicit failure reasons;
- optional preprocessing examples and, only when explicitly enabled,
  `preprocessing_validity_audit.json`;
- validation/test score CSVs, grouped metrics, PRO curves, failure-case and qualitative figures.

Download the complete folder, not just the final printed metric table. The energy JSON is written immediately, even if later accuracy/diagnostic cells are not rerun; an existing latency JSON is updated to include energy. Optional Drive
upload is **disabled** by default. To enable it, install the optional dependencies,
set `DRIVE_UPLOAD_ENABLED=True` and provide Kaggle Secrets `GDRIVE_CLIENT_ID`,
`GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN` and `GDRIVE_FOLDER_ID`.
No TinyGLASS training or sigma re-evaluation is included in this notebook.
